# Chapter 7 — Extending Runs with Events, Hooks, and Explicit Extensions

This chapter adds trusted customization without turning the reusable Runtime into a plugin loader or exposing its internals.

## Goal and Previous Limitation

Chapter 6 can compact and resume a durable Session, but callers still have to edit Runtime code to observe lifecycle barriers, intervene before risky work, or add product-specific behavior. We begin from the immutable Chapter 6 Checkpoint and introduce one bounded Extension interface.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'course' / 'checkpoints' / 'ch06').is_dir():
    raise RuntimeError('run this Chapter Notebook from the repository root')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CHAPTER_6 = ROOT / 'course' / 'checkpoints' / 'ch06'
assert (CHAPTER_6 / 'checkpoint.json').is_file()


## Conceptual Model

`ExtensionHost` is the deep module at the trust seam. Only explicit `Extension` values enter it. Its small `ExtensionAPI` resolves Tools and commands with explicit replacement, preserves subscriber and Hook order, owns lifecycle handlers, and accepts versioned Custom Entries and Run Annotations.

Events are passive: high-level subscribers are awaited barriers but cannot change behavior, while the low-level iterator is an ordered observation stream. Hooks are intervention: the five declared points may replace pending values or fail safe. Every accepted Run captures these effective choices in one immutable `RunSnapshot`.

## Minimal Execution

The following Export Cells add the Extension module and replace only Chapter 6 modules whose interfaces evolve. Each cell owns one complete file.

In [ ]:
EXTENSIONS_SOURCE = '"""Bounded, explicitly supplied Extension registration."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Awaitable, Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nfrom types import MappingProxyType\nimport json\nimport re\n\nfrom .tools import Tool\n\n\nExtensionConfigurator = Callable[["ExtensionAPI"], None]\nEventSubscriber = Callable[[object], Awaitable[None]]\nCommandHandler = Callable[[str], Awaitable[object]]\nLifecycleHandler = Callable[[], Awaitable[None]]\n_COMMAND_NAME = re.compile(r"[a-z][a-z0-9_-]{0,63}\\Z")\n_NAMESPACE = re.compile(\n    r"[a-z][a-z0-9_-]{0,63}(?:\\.[a-z][a-z0-9_-]{0,63})*\\Z"\n)\n\n\n@dataclass(frozen=True, slots=True)\nclass SubscriberRegistration:\n    extension_name: str\n    callback: EventSubscriber\n\n\nclass HookPoint(str, Enum):\n    BEFORE_RUN = "before_run"\n    BEFORE_MODEL_REQUEST = "before_model_request"\n    BEFORE_TOOL_CALL = "before_tool_call"\n    AFTER_TOOL_CALL = "after_tool_call"\n    BEFORE_COMPACTION = "before_compaction"\n\n\n@dataclass(frozen=True, slots=True)\nclass HookContext:\n    point: HookPoint\n    value: object\n    snapshot: object | None = None\n\n\nHookCallback = Callable[[HookContext], Awaitable[object | None]]\n\n\n@dataclass(frozen=True, slots=True)\nclass HookRegistration:\n    extension_name: str\n    point: HookPoint\n    callback: HookCallback\n\n\nclass HookExecutionError(RuntimeError):\n    def __init__(self, extension_name: str, point: HookPoint, detail: str) -> None:\n        self.extension_name = extension_name\n        self.point = point\n        super().__init__(\n            f"Extension {extension_name!r} {point.value} Hook failed: {detail}"\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionHookRequest:\n    trigger: object\n    focus: str | None\n\n\n@dataclass(frozen=True, slots=True)\nclass CommandRegistration:\n    extension_name: str\n    name: str\n    handler: CommandHandler\n\n\n@dataclass(frozen=True, slots=True)\nclass ReplacementRecord:\n    kind: str\n    name: str\n    previous_owner: str\n    replacement_owner: str\n\n\nclass CommandExecutionError(RuntimeError):\n    def __init__(self, command: str, detail: str) -> None:\n        self.command = command\n        super().__init__(f"Command /{command} failed: {detail}")\n\n\n@dataclass(frozen=True, slots=True)\nclass CustomEntry:\n    namespace: str\n    version: int\n    payload: Mapping[str, object]\n\n    def __post_init__(self) -> None:\n        if not _NAMESPACE.fullmatch(self.namespace):\n            raise ValueError("Custom Entry namespace must be a safe dotted name")\n        if (\n            isinstance(self.version, bool)\n            or not isinstance(self.version, int)\n            or self.version < 1\n        ):\n            raise ValueError("Custom Entry version must be a positive integer")\n        try:\n            canonical = json.loads(\n                json.dumps(\n                    dict(self.payload),\n                    ensure_ascii=False,\n                    allow_nan=False,\n                    sort_keys=True,\n                )\n            )\n        except (TypeError, ValueError) as error:\n            raise ValueError("Custom Entry payload must be JSON-serializable") from error\n        if not isinstance(canonical, dict):\n            raise ValueError("Custom Entry payload must be a JSON object")\n        object.__setattr__(self, "payload", MappingProxyType(canonical))\n\n\n@dataclass(frozen=True, slots=True)\nclass RunAnnotation:\n    run_id: str\n    namespace: str\n    version: int\n    payload: Mapping[str, object]\n\n    def __post_init__(self) -> None:\n        if not self.run_id.strip():\n            raise ValueError("Run Annotation requires a Run id")\n        custom = CustomEntry(self.namespace, self.version, self.payload)\n        object.__setattr__(self, "payload", custom.payload)\n\n\nCustomEntrySink = Callable[[CustomEntry], None]\n\n\nclass _ExtensionState:\n    def __init__(self) -> None:\n        self.entries: list[CustomEntry] = []\n        self.pending: list[CustomEntry] = []\n        self.sink: CustomEntrySink | None = None\n        self.annotations: list[RunAnnotation] = []\n\n    def append(self, entry: CustomEntry) -> None:\n        self.entries.append(entry)\n        if self.sink is None:\n            self.pending.append(entry)\n        else:\n            self.sink(entry)\n\n    def activate(self, sink: CustomEntrySink) -> None:\n        self.sink = sink\n        pending = tuple(self.pending)\n        self.pending.clear()\n        for entry in pending:\n            sink(entry)\n\n    def deactivate(self) -> None:\n        self.sink = None\n\n\nclass ExtensionInitializationError(RuntimeError):\n    """Identify an Extension that could not be configured."""\n\n    def __init__(self, source: str, detail: str) -> None:\n        self.source = source\n        super().__init__(f"Extension {source!r} initialization failed: {detail}")\n\n\n@dataclass(frozen=True, slots=True)\nclass Extension:\n    name: str\n    version: str\n    configure: ExtensionConfigurator\n    source: str | None = None\n    startup: LifecycleHandler | None = None\n    teardown: LifecycleHandler | None = None\n\n    def __post_init__(self) -> None:\n        if not _COMMAND_NAME.fullmatch(self.name):\n            raise ValueError("Extension name must be a safe lowercase identifier")\n        if not self.version.strip():\n            raise ValueError("Extension version cannot be empty")\n        if not callable(self.configure):\n            raise TypeError("Extension configure must be callable")\n        if self.startup is not None and not callable(self.startup):\n            raise TypeError("Extension startup must be callable")\n        if self.teardown is not None and not callable(self.teardown):\n            raise TypeError("Extension teardown must be callable")\n\n    @property\n    def identity(self) -> str:\n        return f"{self.name}@{self.version}"\n\n\nclass ExtensionAPI:\n    """The small registration interface visible to one trusted Extension."""\n\n    def __init__(\n        self,\n        extension_name: str,\n        tools: dict[str, Tool],\n        tool_owners: dict[str, str],\n        subscribers: list[SubscriberRegistration],\n        hooks: list[HookRegistration],\n        commands: dict[str, CommandRegistration],\n        state: _ExtensionState,\n        replacements: list[ReplacementRecord],\n    ) -> None:\n        self._extension_name = extension_name\n        self._tools = tools\n        self._tool_owners = tool_owners\n        self._subscribers = subscribers\n        self._hooks = hooks\n        self._commands = commands\n        self._state = state\n        self._replacements = replacements\n\n    def register_tool(self, tool: Tool, *, replace: bool = False) -> None:\n        if not isinstance(tool, Tool):\n            raise TypeError("Extensions can register only Tool values")\n        if tool.name in self._tools and not replace:\n            raise ValueError(\n                f"Tool {tool.name!r} is already registered; use replace=True"\n            )\n        if tool.name in self._tools:\n            self._replacements.append(\n                ReplacementRecord(\n                    "tool",\n                    tool.name,\n                    self._tool_owners[tool.name],\n                    self._extension_name,\n                )\n            )\n        self._tools[tool.name] = tool\n        self._tool_owners[tool.name] = self._extension_name\n\n    def subscribe(self, subscriber: EventSubscriber) -> None:\n        if not callable(subscriber):\n            raise TypeError("Event subscriber must be callable")\n        self._subscribers.append(\n            SubscriberRegistration(self._extension_name, subscriber)\n        )\n\n    def register_hook(self, point: HookPoint, hook: HookCallback) -> None:\n        if not isinstance(point, HookPoint):\n            raise TypeError("Hook point must be a HookPoint")\n        if not callable(hook):\n            raise TypeError("Hook must be callable")\n        self._hooks.append(HookRegistration(self._extension_name, point, hook))\n\n    def register_command(\n        self,\n        name: str,\n        handler: CommandHandler,\n        *,\n        replace: bool = False,\n    ) -> None:\n        if not _COMMAND_NAME.fullmatch(name):\n            raise ValueError("command name must be a safe lowercase identifier")\n        if not callable(handler):\n            raise TypeError("command handler must be callable")\n        if name in self._commands and not replace:\n            raise ValueError(\n                f"Command {name!r} is already registered; use replace=True"\n            )\n        if name in self._commands:\n            self._replacements.append(\n                ReplacementRecord(\n                    "command",\n                    name,\n                    self._commands[name].extension_name,\n                    self._extension_name,\n                )\n            )\n        self._commands[name] = CommandRegistration(\n            self._extension_name,\n            name,\n            handler,\n        )\n\n    def append_custom_entry(\n        self,\n        namespace: str,\n        version: int,\n        payload: Mapping[str, object],\n    ) -> None:\n        if namespace != self._extension_name and not namespace.startswith(\n            f"{self._extension_name}."\n        ):\n            raise ValueError(\n                "Custom Entry namespace must belong to the registering Extension"\n            )\n        self._state.append(CustomEntry(namespace, version, payload))\n\n    def add_run_annotation(\n        self,\n        run_id: str,\n        namespace: str,\n        version: int,\n        payload: Mapping[str, object],\n    ) -> None:\n        if namespace != self._extension_name and not namespace.startswith(\n            f"{self._extension_name}."\n        ):\n            raise ValueError(\n                "Run Annotation namespace must belong to the registering Extension"\n            )\n        self._state.annotations.append(\n            RunAnnotation(run_id, namespace, version, payload)\n        )\n\n\nclass ExtensionHost:\n    """Resolve explicit Extensions into one immutable effective registration set."""\n\n    def __init__(\n        self,\n        base_tools: Sequence[Tool],\n        extensions: Sequence[Extension],\n    ) -> None:\n        tools = {tool.name: tool for tool in base_tools}\n        tool_owners = {tool.name: "runtime" for tool in base_tools}\n        subscribers: list[SubscriberRegistration] = []\n        hooks: list[HookRegistration] = []\n        commands: dict[str, CommandRegistration] = {}\n        state = _ExtensionState()\n        replacements: list[ReplacementRecord] = []\n        identities: list[str] = []\n        names: set[str] = set()\n        for extension in extensions:\n            if not isinstance(extension, Extension):\n                raise TypeError("extensions must contain Extension values")\n            if extension.name in names:\n                raise ValueError(f"duplicate Extension name: {extension.name!r}")\n            names.add(extension.name)\n            source = extension.source or extension.identity\n            try:\n                extension.configure(\n                    ExtensionAPI(\n                        extension.name,\n                        tools,\n                        tool_owners,\n                        subscribers,\n                        hooks,\n                        commands,\n                        state,\n                        replacements,\n                    )\n                )\n            except Exception as error:\n                raise ExtensionInitializationError(\n                    source, f"{type(error).__name__}: {error}"\n                ) from error\n            identities.append(extension.identity)\n        self._tools = MappingProxyType(dict(tools))\n        self._extensions = tuple(extensions)\n        self._identities = tuple(identities)\n        self._subscribers = tuple(subscribers)\n        self._hooks = tuple(hooks)\n        self._commands = MappingProxyType(dict(commands))\n        self._state = state\n        self._started = False\n        self._replacements = tuple(replacements)\n\n    @property\n    def tools(self) -> tuple[Tool, ...]:\n        return tuple(self._tools.values())\n\n    @property\n    def extensions(self) -> tuple[Extension, ...]:\n        return self._extensions\n\n    @property\n    def identities(self) -> tuple[str, ...]:\n        return self._identities\n\n    @property\n    def subscribers(self) -> tuple[SubscriberRegistration, ...]:\n        return self._subscribers\n\n    @property\n    def hooks(self) -> tuple[HookRegistration, ...]:\n        return self._hooks\n\n    @property\n    def commands(self) -> tuple[CommandRegistration, ...]:\n        return tuple(self._commands.values())\n\n    @property\n    def replacements(self) -> tuple[ReplacementRecord, ...]:\n        return self._replacements\n\n    def command(self, name: str) -> CommandRegistration:\n        try:\n            return self._commands[name]\n        except KeyError:\n            raise KeyError(f"unknown chat command: /{name}") from None\n\n    @property\n    def custom_entries(self) -> tuple[CustomEntry, ...]:\n        return tuple(self._state.entries)\n\n    def activate_custom_entry_sink(self, sink: CustomEntrySink) -> None:\n        self._state.activate(sink)\n\n    def deactivate_custom_entry_sink(self) -> None:\n        self._state.deactivate()\n\n    def run_annotations(self, run_id: str | None = None) -> tuple[RunAnnotation, ...]:\n        return tuple(\n            annotation\n            for annotation in self._state.annotations\n            if run_id is None or annotation.run_id == run_id\n        )\n\n    async def startup(self) -> None:\n        if self._started:\n            return\n        for extension in self._extensions:\n            if extension.startup is None:\n                continue\n            source = extension.source or extension.identity\n            try:\n                await extension.startup()\n            except Exception as error:\n                raise ExtensionInitializationError(\n                    source,\n                    f"startup {type(error).__name__}",\n                ) from error\n        self._started = True\n\n    async def teardown(self) -> tuple["LifecycleWarning", ...]:\n        warnings: list[LifecycleWarning] = []\n        for extension in reversed(self._extensions):\n            if extension.teardown is None:\n                continue\n            try:\n                await extension.teardown()\n            except Exception as error:\n                warnings.append(\n                    LifecycleWarning(\n                        extension.source or extension.identity,\n                        "teardown",\n                        type(error).__name__,\n                    )\n                )\n        self._started = False\n        return tuple(warnings)\n\n\n@dataclass(frozen=True, slots=True)\nclass LifecycleWarning:\n    source: str\n    phase: str\n    diagnostic: str\n\n\n@dataclass(frozen=True, slots=True)\nclass ReloadResult:\n    replaced_extensions: tuple[str, ...]\n    warnings: tuple[LifecycleWarning, ...] = ()\n    registration_replacements: tuple[ReplacementRecord, ...] = ()\n'


In [ ]:
MODEL_SOURCE = '"""Provider-neutral message and scripted model contracts for Chapter 3 work."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import AsyncIterator, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom types import MappingProxyType\nfrom typing import Any, Literal, Protocol, TypeAlias, cast\nfrom urllib.parse import urlparse\n\nimport openai\n\n\nclass Role(str, Enum):\n    SYSTEM = "system"\n    USER = "user"\n    ASSISTANT = "assistant"\n    TOOL = "tool"\n\n\nclass StopReason(str, Enum):\n    COMPLETE = "complete"\n    TOOL_USE = "tool_use"\n    LENGTH = "length"\n    CONTENT_FILTER = "content_filter"\n    ERROR = "error"\n    ABORTED = "aborted"\n    OTHER = "other"\n\n\nclass ModelOperation(str, Enum):\n    RUN = "run"\n    COMPACTION = "compaction"\n\n\nclass ModelErrorCode(str, Enum):\n    AUTHENTICATION = "authentication"\n    REQUEST = "request"\n    SCHEMA = "schema"\n    RATE_LIMIT = "rate_limit"\n    TIMEOUT = "timeout"\n    CONNECTION = "connection"\n    SERVER = "server"\n    PROVIDER = "provider"\n    CONTEXT_OVERFLOW = "context_overflow"\n    COMPACTION_FAILED = "compaction_failed"\n    HOOK_FAILED = "hook_failed"\n\n\nclass UnsupportedContentError(ValueError):\n    """Raised before provider I/O for unsupported content."""\n\n\nclass ModelProtocolError(RuntimeError):\n    """Raised when an adapter violates the provider-neutral stream contract."""\n\n\n@dataclass(frozen=True, slots=True)\nclass TextContent:\n    text: str\n    type: Literal["text"] = field(default="text", init=False)\n    schema_version: Literal[1] = field(default=1, init=False)\n\n    def __post_init__(self) -> None:\n        if not isinstance(self.text, str):\n            raise TypeError("TextContent.text must be a string")\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolCallContent:\n    id: str\n    name: str\n    arguments: str\n    type: Literal["tool_call"] = field(default="tool_call", init=False)\n    schema_version: Literal[1] = field(default=1, init=False)\n\n    def __post_init__(self) -> None:\n        if not self.id or not self.name:\n            raise ValueError("a Tool Call requires non-empty id and name")\n        if not isinstance(self.arguments, str):\n            raise TypeError("ToolCallContent.arguments must be a JSON string")\n\n\nContentBlock: TypeAlias = TextContent | ToolCallContent\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelMessage:\n    role: Role\n    content: tuple[ContentBlock, ...]\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelToolResultMessage:\n    tool_call_id: str\n    tool_name: str\n    content: str\n    is_error: bool = False\n    role: Literal[Role.TOOL] = field(default=Role.TOOL, init=False)\n\n\nModelContextMessage: TypeAlias = ModelMessage | ModelToolResultMessage\n\n\n@dataclass(frozen=True, slots=True)\nclass AgentMessage:\n    role: Role\n    content: tuple[ContentBlock, ...]\n\n    @classmethod\n    def text(cls, role: Role, text: str) -> AgentMessage:\n        return cls(role=role, content=(TextContent(text),))\n\n    def to_model(self) -> ModelMessage:\n        if self.role is Role.TOOL:\n            raise UnsupportedContentError("Tool results require ToolResultMessage")\n        content = tuple(_validate_content(block, self.role) for block in self.content)\n        return ModelMessage(self.role, content)\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelSpec:\n    model_id: str\n    context_window: int | None = None\n    max_output_tokens: int = 4096\n    supports_tools: bool = True\n\n    def __post_init__(self) -> None:\n        if not self.model_id.strip():\n            raise ValueError("ModelSpec.model_id cannot be empty")\n        if self.context_window is not None and self.context_window <= 0:\n            raise ValueError("context_window must be positive when supplied")\n        if self.max_output_tokens <= 0:\n            raise ValueError("max_output_tokens must be positive")\n\n\n@dataclass(frozen=True, slots=True)\nclass Usage:\n    input_tokens: int\n    output_tokens: int\n    total_tokens: int\n    estimated: bool = False\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolDefinition:\n    name: str\n    description: str\n    input_schema: Mapping[str, object]\n\n    def __post_init__(self) -> None:\n        object.__setattr__(self, "input_schema", MappingProxyType(dict(self.input_schema)))\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelRequest:\n    messages: tuple[ModelContextMessage, ...]\n    model: ModelSpec\n    tools: tuple[ToolDefinition, ...] = ()\n    operation: ModelOperation = ModelOperation.RUN\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelResult:\n    message: ModelMessage\n    stop_reason: StopReason\n    usage: Usage | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelError:\n    code: ModelErrorCode\n    message: str\n    retryable: bool\n    status_code: int | None = None\n    retry_after_seconds: float | None = None\n\n\nclass ModelAdapterError(RuntimeError):\n    def __init__(self, error: ModelError) -> None:\n        self.error = error\n        super().__init__(error.message)\n\n\n@dataclass(frozen=True, slots=True)\nclass TextDelta:\n    text: str\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolCallDelta:\n    index: int\n    id: str = ""\n    name: str = ""\n    arguments_delta: str = ""\n\n\n@dataclass(frozen=True, slots=True)\nclass UsageUpdate:\n    usage: Usage\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelEnd:\n    stop_reason: StopReason\n\n\nModelEvent: TypeAlias = TextDelta | ToolCallDelta | UsageUpdate | ModelEnd\n\n\nclass ModelAdapter(Protocol):\n    def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]: ...\n\n\ndef _validate_content(block: object, role: Role) -> ContentBlock:\n    if not isinstance(block, (TextContent, ToolCallContent)):\n        raise UnsupportedContentError(\n            "version one supports only TextContent and ToolCallContent"\n        )\n    if getattr(block, "schema_version", None) != 1:\n        raise UnsupportedContentError("unsupported Content Block schema version")\n    if isinstance(block, ToolCallContent) and role is not Role.ASSISTANT:\n        raise UnsupportedContentError(\n            "ToolCallContent is valid only for assistant messages"\n        )\n    return block\n\n\ndef to_model_messages(messages: Sequence[object]) -> tuple[ModelContextMessage, ...]:\n    converted: list[ModelContextMessage] = []\n    for message in messages:\n        convert = getattr(message, "to_model", None)\n        if not callable(convert):\n            raise TypeError("conversation messages must provide to_model()")\n        converted.append(convert())\n    return tuple(converted)\n\n\nclass ScriptedModelAdapter:\n    """Replay one or more provider-neutral model turns."""\n\n    def __init__(\n        self,\n        events: Sequence[ModelEvent] | Sequence[Sequence[ModelEvent]],\n    ) -> None:\n        items = tuple(events)\n        if items and isinstance(items[0], (list, tuple)):\n            self._turns = tuple(tuple(turn) for turn in items)  # type: ignore[arg-type]\n        else:\n            self._turns = (items,)  # type: ignore[assignment]\n        self._requests: list[ModelRequest] = []\n\n    @property\n    def received_requests(self) -> tuple[ModelRequest, ...]:\n        return tuple(self._requests)\n\n    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:\n        turn = len(self._requests)\n        self._requests.append(request)\n        if turn >= len(self._turns):\n            raise RuntimeError("script has no model turn remaining")\n        for event in self._turns[turn]:\n            yield event\n\n\nasync def complete(\n    adapter: ModelAdapter,\n    messages: Sequence[AgentMessage],\n    model: ModelSpec,\n) -> ModelResult:\n    request = ModelRequest(to_model_messages(messages), model)\n    text_parts: list[str] = []\n    tool_drafts: dict[int, dict[str, str]] = {}\n    usage: Usage | None = None\n    end: ModelEnd | None = None\n    async for event in adapter.stream(request):\n        if end is not None:\n            raise ModelProtocolError("an adapter emitted data after ModelEnd")\n        if isinstance(event, TextDelta):\n            text_parts.append(event.text)\n        elif isinstance(event, ToolCallDelta):\n            if event.index < 0:\n                raise ModelProtocolError("Tool Call indexes cannot be negative")\n            draft = tool_drafts.setdefault(\n                event.index, {"id": "", "name": "", "arguments": ""}\n            )\n            draft["id"] += event.id\n            draft["name"] += event.name\n            draft["arguments"] += event.arguments_delta\n        elif isinstance(event, UsageUpdate):\n            usage = event.usage\n        elif isinstance(event, ModelEnd):\n            end = event\n        else:\n            raise ModelProtocolError(\n                f"unsupported model event: {type(event).__name__}"\n            )\n    if end is None:\n        raise ModelProtocolError("an adapter stream must end with ModelEnd")\n    blocks: list[ContentBlock] = []\n    if text_parts:\n        blocks.append(TextContent("".join(text_parts)))\n    for index in sorted(tool_drafts):\n        draft = tool_drafts[index]\n        try:\n            blocks.append(ToolCallContent(**draft))\n        except (TypeError, ValueError) as error:\n            raise ModelProtocolError(f"incomplete Tool Call at index {index}") from error\n    return ModelResult(\n        message=ModelMessage(Role.ASSISTANT, tuple(blocks)),\n        stop_reason=end.stop_reason,\n        usage=usage,\n    )\n\n\n@dataclass(frozen=True, slots=True)\nclass OpenAICompatibleConfig:\n    base_url: str\n    api_key: str\n    headers: Mapping[str, str] = field(default_factory=dict)\n    extra_body: Mapping[str, object] = field(default_factory=dict)\n    timeout_seconds: float = 60.0\n\n    def __post_init__(self) -> None:\n        parsed = urlparse(self.base_url)\n        if parsed.scheme not in {"http", "https"} or not parsed.netloc:\n            raise ValueError("base_url must be an explicit HTTP(S) URL")\n        if not self.api_key:\n            raise ValueError("api_key must be supplied explicitly")\n        if self.timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive")\n        object.__setattr__(self, "headers", MappingProxyType(dict(self.headers)))\n        object.__setattr__(self, "extra_body", MappingProxyType(dict(self.extra_body)))\n\n\ndef _provider_message(message: ModelContextMessage) -> dict[str, object]:\n    if isinstance(message, ModelToolResultMessage):\n        return {\n            "role": "tool",\n            "tool_call_id": message.tool_call_id,\n            "content": message.content,\n        }\n    text = "".join(\n        block.text for block in message.content if isinstance(block, TextContent)\n    )\n    tool_calls = [\n        block for block in message.content if isinstance(block, ToolCallContent)\n    ]\n    encoded: dict[str, object] = {\n        "role": message.role.value,\n        "content": text or None,\n    }\n    if tool_calls:\n        encoded["tool_calls"] = [\n            {\n                "id": block.id,\n                "type": "function",\n                "function": {"name": block.name, "arguments": block.arguments},\n            }\n            for block in tool_calls\n        ]\n    return encoded\n\n\ndef _provider_tool(tool: ToolDefinition) -> dict[str, object]:\n    return {\n        "type": "function",\n        "function": {\n            "name": tool.name,\n            "description": tool.description,\n            "parameters": dict(tool.input_schema),\n        },\n    }\n\n\ndef _stop_reason(value: str | None) -> StopReason:\n    if value is None:\n        return StopReason.OTHER\n    return {\n        "stop": StopReason.COMPLETE,\n        "tool_calls": StopReason.TOOL_USE,\n        "length": StopReason.LENGTH,\n        "content_filter": StopReason.CONTENT_FILTER,\n    }.get(value, StopReason.OTHER)\n\n\ndef _normalized_error(error: Exception) -> ModelError:\n    status = getattr(error, "status_code", None)\n    raw_body = getattr(error, "body", None)\n    provider_markers: list[str] = []\n    for value in (\n        getattr(error, "code", None),\n        getattr(error, "type", None),\n        getattr(error, "message", None),\n    ):\n        if isinstance(value, str):\n            provider_markers.append(value.lower())\n    if isinstance(raw_body, dict):\n        for key in ("code", "type", "message"):\n            value = raw_body.get(key)\n            if isinstance(value, str):\n                provider_markers.append(value.lower())\n    context_overflow = status == 400 and any(\n        marker in value\n        for value in provider_markers\n        for marker in (\n            "context_length_exceeded",\n            "context_window_exceeded",\n            "maximum context length",\n            "context window is too long",\n        )\n    )\n    retry_after_seconds: float | None = None\n    response = getattr(error, "response", None)\n    headers = getattr(response, "headers", None)\n    if headers is not None:\n        raw_retry_after = headers.get("retry-after")\n        if raw_retry_after is not None:\n            try:\n                parsed_retry_after = float(raw_retry_after)\n            except (TypeError, ValueError):\n                pass\n            else:\n                if parsed_retry_after >= 0:\n                    retry_after_seconds = parsed_retry_after\n    if context_overflow:\n        code, retryable = ModelErrorCode.CONTEXT_OVERFLOW, False\n    elif isinstance(error, openai.AuthenticationError):\n        code, retryable = ModelErrorCode.AUTHENTICATION, False\n    elif isinstance(error, openai.RateLimitError) or status == 429:\n        code, retryable = ModelErrorCode.RATE_LIMIT, True\n    elif isinstance(error, openai.APITimeoutError) or status == 408:\n        code, retryable = ModelErrorCode.TIMEOUT, True\n    elif isinstance(error, openai.APIConnectionError):\n        code, retryable = ModelErrorCode.CONNECTION, True\n    elif isinstance(status, int) and status >= 500:\n        code, retryable = ModelErrorCode.SERVER, True\n    elif isinstance(error, (openai.BadRequestError, openai.NotFoundError)):\n        code, retryable = ModelErrorCode.REQUEST, False\n    else:\n        code, retryable = ModelErrorCode.PROVIDER, False\n    status_text = f" with status {status}" if status is not None else ""\n    return ModelError(\n        code=code,\n        message=f"OpenAI-compatible request failed{status_text}",\n        retryable=retryable,\n        status_code=status,\n        retry_after_seconds=retry_after_seconds,\n    )\n\n\nclass OpenAICompatibleAdapter:\n    """Translate streaming Chat Completions at the ModelAdapter seam."""\n\n    def __init__(self, config: OpenAICompatibleConfig) -> None:\n        self._config = config\n        self._client = openai.AsyncOpenAI(\n            api_key=config.api_key,\n            base_url=config.base_url.rstrip("/") + "/",\n            default_headers=dict(config.headers),\n            timeout=config.timeout_seconds,\n            max_retries=0,\n        )\n\n    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:\n        finish_reason: str | None = None\n        try:\n            response = await self._client.chat.completions.create(\n                model=request.model.model_id,\n                messages=cast(list, [_provider_message(item) for item in request.messages]),\n                max_tokens=request.model.max_output_tokens,\n                tools=cast(\n                    Any,\n                    [_provider_tool(tool) for tool in request.tools]\n                    if request.tools\n                    else openai.NOT_GIVEN,\n                ),\n                stream=True,\n                stream_options={"include_usage": True},\n                extra_body=dict(self._config.extra_body) or None,\n            )\n            async for chunk in response:\n                if chunk.usage is not None:\n                    input_tokens = chunk.usage.prompt_tokens or 0\n                    output_tokens = chunk.usage.completion_tokens or 0\n                    total_tokens = (\n                        chunk.usage.total_tokens or input_tokens + output_tokens\n                    )\n                    yield UsageUpdate(\n                        Usage(input_tokens, output_tokens, total_tokens)\n                    )\n                for choice in chunk.choices:\n                    delta = choice.delta\n                    if delta.content:\n                        yield TextDelta(delta.content)\n                    for tool_call in delta.tool_calls or ():\n                        function = tool_call.function\n                        yield ToolCallDelta(\n                            index=tool_call.index,\n                            id=tool_call.id or "",\n                            name=(function.name if function else None) or "",\n                            arguments_delta=(\n                                function.arguments if function else None\n                            )\n                            or "",\n                        )\n                    if choice.finish_reason is not None:\n                        finish_reason = choice.finish_reason\n        except openai.OpenAIError as error:\n            raise ModelAdapterError(_normalized_error(error)) from None\n        yield ModelEnd(_stop_reason(finish_reason))\n'


In [ ]:
TOOLS_SOURCE = '"""Explicit Tools and their replaceable execution seam."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import Awaitable, Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom types import MappingProxyType\nfrom typing import Protocol\nfrom pathlib import Path\n\nfrom jsonschema import Draft202012Validator  # type: ignore[import-untyped]\n\nfrom .model import ModelToolResultMessage, ToolCallContent, ToolDefinition\n\n\nToolHandler = Callable[[dict[str, object]], Awaitable["ToolResult"]]\n\n\nclass ToolErrorCode(str, Enum):\n    INVALID_JSON = "invalid_json"\n    INVALID_ARGUMENTS = "invalid_arguments"\n    UNKNOWN_TOOL = "unknown_tool"\n    EXECUTION_FAILED = "execution_failed"\n    CANCELLED = "cancelled"\n    HOOK_FAILED = "hook_failed"\n\n\nclass TruncationDirection(str, Enum):\n    HEAD = "head"\n    TAIL = "tail"\n\n\nclass CompleteOutputKind(str, Enum):\n    ARTIFACT = "artifact"\n    EXTERNAL = "external"\n    UNAVAILABLE = "unavailable"\n\n\n@dataclass(frozen=True, slots=True)\nclass CompleteOutputReference:\n    kind: CompleteOutputKind\n    reference: str | None = None\n    reason: str | None = None\n\n    @classmethod\n    def artifact(cls, reference: str) -> "CompleteOutputReference":\n        return cls(CompleteOutputKind.ARTIFACT, reference=reference)\n\n    @classmethod\n    def unavailable(cls) -> "CompleteOutputReference":\n        return cls(\n            CompleteOutputKind.UNAVAILABLE,\n            reason="complete output was not retained",\n        )\n\n    def render(self) -> str:\n        if self.reference is not None:\n            return f"{self.kind.value} {self.reference}"\n        return f"{self.kind.value} ({self.reason})"\n\n\n@dataclass(frozen=True, slots=True)\nclass TruncationNotice:\n    original_bytes: int\n    original_lines: int\n    retained_start_byte: int\n    retained_end_byte: int\n    retained_start_line: int\n    retained_end_line: int\n    direction: TruncationDirection\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolOutputBudget:\n    max_bytes: int = 50 * 1024\n    max_lines: int = 2000\n\n    def __post_init__(self) -> None:\n        if self.max_bytes <= 0 or self.max_lines <= 0:\n            raise ValueError("Tool output limits must be positive")\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolResult:\n    content: str\n    metadata: Mapping[str, object] = field(default_factory=dict)\n    terminate: bool = False\n    is_error: bool = False\n    error_code: ToolErrorCode | None = None\n    truncation: TruncationNotice | None = None\n    complete_output: CompleteOutputReference | None = None\n\n    def __post_init__(self) -> None:\n        object.__setattr__(self, "metadata", MappingProxyType(dict(self.metadata)))\n        if self.is_error != (self.error_code is not None):\n            raise ValueError("error ToolResult values require exactly one error_code")\n        if self.is_error and self.terminate:\n            raise ValueError("error ToolResult values cannot terminate a Tool Batch")\n\n    @classmethod\n    def error(cls, code: ToolErrorCode, tool_name: str) -> "ToolResult":\n        guidance = {\n            ToolErrorCode.INVALID_JSON: "provide one valid JSON object",\n            ToolErrorCode.INVALID_ARGUMENTS: "match the Tool\'s declared input schema",\n            ToolErrorCode.UNKNOWN_TOOL: "choose one of the advertised Tools",\n            ToolErrorCode.EXECUTION_FAILED: "revise the call or choose another Tool",\n            ToolErrorCode.CANCELLED: "retry after the cancelled Run is settled",\n            ToolErrorCode.HOOK_FAILED: "review the Extension Hook failure before retrying",\n        }[code]\n        return cls(\n            content=f"Tool error [{code.value}] for \'{tool_name}\': {guidance}.",\n            is_error=True,\n            error_code=code,\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass Tool:\n    name: str\n    description: str\n    input_schema: Mapping[str, object]\n    execute: ToolHandler\n    sequential: bool = False\n    output_direction: TruncationDirection = TruncationDirection.HEAD\n\n    def __post_init__(self) -> None:\n        if not self.name.strip():\n            raise ValueError("Tool name cannot be empty")\n        if not self.description.strip():\n            raise ValueError("Tool description cannot be empty")\n        if not callable(self.execute):\n            raise TypeError("Tool execute must be an async callable")\n        schema = dict(self.input_schema)\n        Draft202012Validator.check_schema(schema)\n        if schema.get("type") != "object":\n            raise ValueError("Tool input_schema must describe a JSON object")\n        object.__setattr__(self, "input_schema", MappingProxyType(schema))\n\n    def definition(self) -> ToolDefinition:\n        return ToolDefinition(self.name, self.description, self.input_schema)\n\n\n@dataclass(frozen=True, slots=True)\nclass PreparedToolCall:\n    call: ToolCallContent\n    tool: Tool\n    arguments: dict[str, object]\n\n\n@dataclass(frozen=True, slots=True)\nclass ToolResultMessage:\n    tool_call_id: str\n    tool_name: str\n    result: ToolResult\n\n    def to_model(self) -> ModelToolResultMessage:\n        return ModelToolResultMessage(\n            self.tool_call_id,\n            self.tool_name,\n            self.result.content,\n            self.result.is_error,\n        )\n\n\nclass ToolExecutor(Protocol):\n    async def execute(self, call: PreparedToolCall) -> ToolResult: ...\n\n\nclass LocalToolExecutor:\n    async def execute(self, call: PreparedToolCall) -> ToolResult:\n        return await call.tool.execute(call.arguments)\n\n\n@dataclass(frozen=True, slots=True)\nclass ProcessResult:\n    returncode: int\n    stdout: str\n    stderr: str\n\n\nclass AsyncioProcessOperations:\n    """Run an argv directly and terminate the child when its task is cancelled.\n\n    This is host-process execution infrastructure, not a sandbox.\n    """\n\n    async def run(\n        self,\n        command: Sequence[str],\n        *,\n        cwd: str | Path | None = None,\n        timeout_seconds: float | None = None,\n    ) -> ProcessResult:\n        argv = tuple(command)\n        if not argv or any(not isinstance(part, str) or not part for part in argv):\n            raise ValueError("command must contain non-empty argv strings")\n        if timeout_seconds is not None and timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive when supplied")\n        process = await asyncio.create_subprocess_exec(\n            *argv,\n            cwd=cwd,\n            stdout=asyncio.subprocess.PIPE,\n            stderr=asyncio.subprocess.PIPE,\n        )\n        try:\n            communication = process.communicate()\n            if timeout_seconds is None:\n                stdout, stderr = await communication\n            else:\n                stdout, stderr = await asyncio.wait_for(\n                    communication, timeout_seconds\n                )\n        except (asyncio.CancelledError, TimeoutError):\n            if process.returncode is None:\n                process.terminate()\n                try:\n                    await asyncio.wait_for(process.wait(), timeout=1)\n                except TimeoutError:\n                    process.kill()\n                    await process.wait()\n            raise\n        assert process.returncode is not None\n        return ProcessResult(\n            process.returncode,\n            stdout.decode("utf-8", errors="replace"),\n            stderr.decode("utf-8", errors="replace"),\n        )\n\n\ndef bound_tool_result(\n    result: ToolResult,\n    budget: ToolOutputBudget,\n    direction: TruncationDirection,\n) -> ToolResult:\n    """Bound model-facing text and attach explicit recovery provenance."""\n\n    content = result.content\n    encoded = content.encode("utf-8")\n    lines = content.splitlines(keepends=True)\n    original_lines = len(content.splitlines())\n    if len(encoded) <= budget.max_bytes and original_lines <= budget.max_lines:\n        return result\n\n    if direction is TruncationDirection.HEAD:\n        line_limited = "".join(lines[: budget.max_lines])\n        retained_bytes = line_limited.encode("utf-8")[: budget.max_bytes]\n        retained = retained_bytes.decode("utf-8", errors="ignore")\n        retained_start_byte = 0\n        retained_end_byte = len(retained.encode("utf-8"))\n        retained_start_line = 1 if retained else 0\n        retained_end_line = len(retained.splitlines())\n    else:\n        line_limited = "".join(lines[-budget.max_lines :])\n        retained_bytes = line_limited.encode("utf-8")[-budget.max_bytes :]\n        retained = retained_bytes.decode("utf-8", errors="ignore")\n        retained_end_byte = len(encoded)\n        retained_start_byte = retained_end_byte - len(retained.encode("utf-8"))\n        retained_end_line = original_lines\n        retained_line_count = len(retained.splitlines())\n        retained_start_line = max(1, original_lines - retained_line_count + 1)\n\n    reference = result.complete_output or CompleteOutputReference.unavailable()\n    notice = TruncationNotice(\n        original_bytes=len(encoded),\n        original_lines=original_lines,\n        retained_start_byte=retained_start_byte,\n        retained_end_byte=retained_end_byte,\n        retained_start_line=retained_start_line,\n        retained_end_line=retained_end_line,\n        direction=direction,\n    )\n    model_notice = (\n        "[tool output truncated: "\n        f"retained {direction.value} bytes {retained_start_byte}-{retained_end_byte} "\n        f"of {len(encoded)}, lines {retained_start_line}-{retained_end_line} "\n        f"of {original_lines}; complete output: {reference.render()}]"\n    )\n    return ToolResult(\n        content=f"{retained}\\n\\n{model_notice}",\n        metadata=result.metadata,\n        terminate=result.terminate,\n        is_error=result.is_error,\n        error_code=result.error_code,\n        truncation=notice,\n        complete_output=reference,\n    )\n'


In [ ]:
RUNTIME_SOURCE = '"""Async Agent Runtime with structured Tool batches."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import AsyncIterator, Awaitable, Callable, Mapping, Sequence\nfrom dataclasses import dataclass, replace\nfrom enum import Enum\nimport hashlib\nimport json\nfrom types import MappingProxyType\nfrom typing import Protocol, TypeAlias, cast\nfrom uuid import uuid4\n\nfrom jsonschema import (  # type: ignore[import-untyped]\n    Draft202012Validator,\n    ValidationError,\n)\n\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelRequest,\n    ModelOperation,\n    ModelSpec,\n    Role,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    Usage,\n    UsageUpdate,\n    to_model_messages,\n)\nfrom .compaction import CompactionCheckpoint\nfrom .extensions import (\n    HookContext,\n    HookExecutionError,\n    HookPoint,\n    HookRegistration,\n    SubscriberRegistration,\n)\nfrom .tools import (\n    LocalToolExecutor,\n    PreparedToolCall,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    bound_tool_result,\n)\n\n\nclass EventType(str, Enum):\n    AGENT_START = "agent_start"\n    MODEL_ATTEMPT_START = "model_attempt_start"\n    MODEL_EVENT = "model_event"\n    MODEL_ATTEMPT_FAILED = "model_attempt_failed"\n    RETRY_SCHEDULED = "retry_scheduled"\n    COMPACTION_START = "compaction_start"\n    COMPACTION_END = "compaction_end"\n    COMPACTION_FAILED = "compaction_failed"\n    TOOL_BATCH_START = "tool_batch_start"\n    TOOL_CALL_START = "tool_call_start"\n    TOOL_CALL_END = "tool_call_end"\n    TOOL_BATCH_END = "tool_batch_end"\n    RUN_CANCELLED = "run_cancelled"\n    MESSAGE_END = "message_end"\n    AGENT_END = "agent_end"\n    SUBSCRIBER_FAILED = "subscriber_failed"\n\n\nclass TerminalStatus(str, Enum):\n    COMPLETED = "completed"\n    MODEL_ERROR = "model_error"\n    CANCELLED = "cancelled"\n    MAX_TURNS = "max_turns"\n    MAX_TOOL_CALLS = "max_tool_calls"\n    TIMEOUT = "timeout"\n    MAX_TOTAL_TOKENS = "max_total_tokens"\n\n\n@dataclass(frozen=True, slots=True)\nclass RunGuard:\n    max_turns: int | None = None\n    max_tool_calls: int | None = None\n    timeout_seconds: float | None = None\n    max_total_tokens: int | None = None\n\n    def __post_init__(self) -> None:\n        integer_limits = {\n            "max_turns": self.max_turns,\n            "max_tool_calls": self.max_tool_calls,\n            "max_total_tokens": self.max_total_tokens,\n        }\n        for name, value in integer_limits.items():\n            if value is not None and (\n                isinstance(value, bool) or not isinstance(value, int) or value <= 0\n            ):\n                raise ValueError(f"{name} must be a positive integer when supplied")\n        if self.timeout_seconds is not None and self.timeout_seconds <= 0:\n            raise ValueError("timeout_seconds must be positive when supplied")\n\n    def reached(\n        self,\n        *,\n        turns: int,\n        tool_calls: int,\n        usage: Usage | None,\n        elapsed_seconds: float,\n    ) -> TerminalStatus | None:\n        if (\n            self.timeout_seconds is not None\n            and elapsed_seconds >= self.timeout_seconds\n        ):\n            return TerminalStatus.TIMEOUT\n        if self.max_turns is not None and turns >= self.max_turns:\n            return TerminalStatus.MAX_TURNS\n        if (\n            self.max_tool_calls is not None\n            and tool_calls >= self.max_tool_calls\n        ):\n            return TerminalStatus.MAX_TOOL_CALLS\n        if (\n            self.max_total_tokens is not None\n            and usage is not None\n            and usage.total_tokens >= self.max_total_tokens\n        ):\n            return TerminalStatus.MAX_TOTAL_TOKENS\n        return None\n\n\n@dataclass(frozen=True, slots=True)\nclass RuntimeEvent:\n    sequence: int\n    type: EventType\n    attempt: int | None = None\n    model_event: ModelEvent | None = None\n    error: ModelError | None = None\n    retry_delay_seconds: float | None = None\n    partial_text: str = ""\n    partial_usage: Usage | None = None\n    tool_call_id: str | None = None\n    tool_name: str | None = None\n    tool_result: ToolResult | None = None\n    operation: ModelOperation = ModelOperation.RUN\n    extension_name: str | None = None\n    diagnostic: str | None = None\n    run_id: str | None = None\n    snapshot_fingerprint: str | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass AssistantOutcome:\n    message: AgentMessage\n    stop_reason: StopReason\n    usage: Usage | None = None\n    error: ModelError | None = None\n    attempts: int = 1\n    tool_results: tuple[ToolResult, ...] = ()\n    status: TerminalStatus = TerminalStatus.COMPLETED\n    snapshot_fingerprint: str | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass SummaryGeneration:\n    text: str\n    usage: Usage | None\n    attempts: int\n\n\n@dataclass(frozen=True, slots=True)\nclass RetryPolicy:\n    delays: tuple[float, ...] = (2.0, 4.0, 8.0)\n    max_retry_after_seconds: float = 60.0\n\n    def __post_init__(self) -> None:\n        if any(delay < 0 for delay in self.delays):\n            raise ValueError("retry delays cannot be negative")\n        if self.max_retry_after_seconds < 0:\n            raise ValueError("max_retry_after_seconds cannot be negative")\n\n    def delay_for(\n        self,\n        error: ModelError,\n        failed_attempt: int,\n        *,\n        retry_after_seconds: float | None = None,\n    ) -> float | None:\n        retryable_codes = {\n            ModelErrorCode.RATE_LIMIT,\n            ModelErrorCode.TIMEOUT,\n            ModelErrorCode.CONNECTION,\n            ModelErrorCode.SERVER,\n        }\n        retryable_status = error.status_code in {408, 429} or (\n            error.status_code is not None and error.status_code >= 500\n        )\n        if (\n            error.code not in retryable_codes and not retryable_status\n        ) or failed_attempt > len(self.delays):\n            return None\n        if retry_after_seconds is not None:\n            if not 0 <= retry_after_seconds <= self.max_retry_after_seconds:\n                return None\n            return retry_after_seconds\n        return self.delays[failed_attempt - 1]\n\n\n@dataclass(frozen=True, slots=True)\nclass RunSnapshot:\n    """Immutable effective behavior captured when a Run is accepted."""\n\n    run_id: str\n    adapter: ModelAdapter\n    model: ModelSpec\n    tools: tuple[Tool, ...]\n    tool_executor: ToolExecutor\n    tool_output_budget: ToolOutputBudget\n    retry_policy: RetryPolicy\n    sleeper: Sleeper\n    run_guard: object | None\n    extension_identities: tuple[str, ...]\n    subscribers: tuple[SubscriberRegistration, ...]\n    hooks: tuple[HookRegistration, ...]\n    generation_settings: Mapping[str, object]\n    compaction_policy: object | None\n    prompt_hashes: tuple[tuple[str, str], ...]\n    resource_hashes: tuple[tuple[str, str], ...]\n    fingerprint: str\n\n    @classmethod\n    def capture(\n        cls,\n        *,\n        adapter: ModelAdapter,\n        model: ModelSpec,\n        tools: Sequence[Tool],\n        tool_executor: ToolExecutor,\n        tool_output_budget: ToolOutputBudget,\n        retry_policy: RetryPolicy,\n        sleeper: Sleeper,\n        run_guard: object | None,\n        extension_identities: Sequence[str] = (),\n        subscribers: Sequence[SubscriberRegistration] = (),\n        hooks: Sequence[HookRegistration] = (),\n        generation_settings: Mapping[str, object] | None = None,\n        compaction_policy: object | None = None,\n        prompt_hashes: Mapping[str, str] | None = None,\n        resource_hashes: Mapping[str, str] | None = None,\n    ) -> "RunSnapshot":\n        accepted_tools = tuple(tools)\n        accepted_extensions = tuple(extension_identities)\n        accepted_subscribers = tuple(subscribers)\n        accepted_hooks = tuple(hooks)\n        accepted_generation = dict(generation_settings or {})\n        accepted_prompt_hashes = tuple(sorted((prompt_hashes or {}).items()))\n        accepted_resource_hashes = tuple(sorted((resource_hashes or {}).items()))\n        payload = {\n            "model": {\n                "id": model.model_id,\n                "context_window": model.context_window,\n                "max_output_tokens": model.max_output_tokens,\n                "supports_tools": model.supports_tools,\n            },\n            "tools": [\n                {\n                    "name": tool.name,\n                    "description": tool.description,\n                    "schema": dict(tool.input_schema),\n                    "sequential": tool.sequential,\n                    "output_direction": tool.output_direction.value,\n                }\n                for tool in accepted_tools\n            ],\n            "extensions": list(accepted_extensions),\n            "hooks": [\n                [registration.extension_name, registration.point.value]\n                for registration in accepted_hooks\n            ],\n            "generation": accepted_generation,\n            "retry": {\n                "delays": retry_policy.delays,\n                "max_retry_after_seconds": retry_policy.max_retry_after_seconds,\n            },\n            "compaction": repr(compaction_policy),\n            "guard": repr(run_guard),\n            "prompt_hashes": accepted_prompt_hashes,\n            "resource_hashes": accepted_resource_hashes,\n        }\n        canonical = json.dumps(\n            payload,\n            ensure_ascii=False,\n            sort_keys=True,\n            separators=(",", ":"),\n        ).encode("utf-8")\n        return cls(\n            uuid4().hex,\n            adapter,\n            model,\n            accepted_tools,\n            tool_executor,\n            tool_output_budget,\n            retry_policy,\n            sleeper,\n            run_guard,\n            accepted_extensions,\n            accepted_subscribers,\n            accepted_hooks,\n            MappingProxyType(accepted_generation),\n            compaction_policy,\n            accepted_prompt_hashes,\n            accepted_resource_hashes,\n            hashlib.sha256(canonical).hexdigest(),\n        )\n\n\nSleeper: TypeAlias = Callable[[float], Awaitable[None]]\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\nSettlementSink: TypeAlias = Callable[[Sequence[ConversationMessage]], None]\nContextOverflowRecovery: TypeAlias = Callable[[], Awaitable[bool]]\nSettledTurnHandler: TypeAlias = Callable[[], Awaitable[object]]\n_EVENTS_DONE = object()\n\n\nclass TurnInput(Protocol):\n    """Runtime-facing view of Steering Messages waiting at a turn boundary."""\n\n    def pending(self) -> bool: ...\n\n    def take(self) -> Sequence[AgentMessage]: ...\n\n\n@dataclass(slots=True)\nclass _CancellationState:\n    status: TerminalStatus = TerminalStatus.CANCELLED\n    requested: bool = False\n\n    def request(self, status: TerminalStatus) -> bool:\n        if self.requested:\n            return False\n        self.status = status\n        self.requested = True\n        return True\n\n\ndef _add_usage(left: Usage | None, right: Usage | None) -> Usage | None:\n    if left is None:\n        return right\n    if right is None:\n        return left\n    return Usage(\n        left.input_tokens + right.input_tokens,\n        left.output_tokens + right.output_tokens,\n        left.total_tokens + right.total_tokens,\n        left.estimated or right.estimated,\n    )\n\n\nclass AgentRunHandle:\n    """One accepted run\'s observations, cancellation, and eventual outcome."""\n\n    def __init__(\n        self,\n        task: asyncio.Task[AssistantOutcome],\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n        snapshot: RunSnapshot,\n    ) -> None:\n        self._task = task\n        self._events = events\n        self._cancellation = cancellation\n        self._snapshot = snapshot\n\n    @property\n    def run_id(self) -> str:\n        return self._snapshot.run_id\n\n    @property\n    def snapshot(self) -> RunSnapshot:\n        return self._snapshot\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n    async def result(self) -> AssistantOutcome:\n        outcome = await self._task\n        if outcome.snapshot_fingerprint == self._snapshot.fingerprint:\n            return outcome\n        return replace(\n            outcome,\n            snapshot_fingerprint=self._snapshot.fingerprint,\n        )\n\n    def cancel(self) -> None:\n        self._cancel_with(TerminalStatus.CANCELLED)\n\n    def _cancel_with(self, status: TerminalStatus) -> None:\n        if not self._task.done() and self._cancellation.request(status):\n            self._task.get_loop().call_soon(self._task.cancel)\n\n\nclass AgentRuntime:\n    """Advance typed conversation state through model and Tool turns."""\n\n    def __init__(\n        self,\n        adapter: ModelAdapter,\n        model: ModelSpec,\n        *,\n        tools: Sequence[Tool] = (),\n        tool_executor: ToolExecutor | None = None,\n        tool_output_budget: ToolOutputBudget | None = None,\n        retry_policy: RetryPolicy | None = None,\n        sleeper: Sleeper = asyncio.sleep,\n        run_guard: object | None = None,\n        generation_settings: Mapping[str, object] | None = None,\n        history: Sequence[ConversationMessage] = (),\n    ) -> None:\n        if not isinstance(model, ModelSpec):\n            raise TypeError("model must be a ModelSpec")\n        if not callable(getattr(adapter, "stream", None)):\n            raise TypeError("adapter must implement ModelAdapter.stream")\n        if retry_policy is not None and not isinstance(retry_policy, RetryPolicy):\n            raise TypeError("retry_policy must be a RetryPolicy")\n        if not callable(sleeper):\n            raise TypeError("sleeper must be an async callable")\n        if tool_executor is not None and not callable(\n            getattr(tool_executor, "execute", None)\n        ):\n            raise TypeError("tool_executor must implement ToolExecutor.execute")\n        if tool_output_budget is not None and not isinstance(\n            tool_output_budget, ToolOutputBudget\n        ):\n            raise TypeError("tool_output_budget must be a ToolOutputBudget")\n        registered: dict[str, Tool] = {}\n        for tool in tools:\n            if not isinstance(tool, Tool):\n                raise TypeError("tools must contain Tool values")\n            if tool.name in registered:\n                raise ValueError(f"duplicate Tool name: {tool.name!r}")\n            registered[tool.name] = tool\n        if registered and not model.supports_tools:\n            raise ValueError("configured ModelSpec does not support Tools")\n        self._adapter = adapter\n        self._model = model\n        self._tools = registered\n        self._tool_executor = tool_executor or LocalToolExecutor()\n        self._tool_output_budget = tool_output_budget or ToolOutputBudget()\n        self._retry_policy = retry_policy or RetryPolicy()\n        self._sleeper = sleeper\n        self._run_guard = run_guard\n        self._generation_settings = MappingProxyType(dict(generation_settings or {}))\n        accepted_history = tuple(history)\n        to_model_messages(accepted_history)\n        self._history: list[ConversationMessage] = list(accepted_history)\n        self._effective_history: list[ConversationMessage] | None = None\n        self._settlement_sink: SettlementSink | None = None\n        self._context_overflow_recovery: ContextOverflowRecovery | None = None\n        self._settled_turn_handler: SettledTurnHandler | None = None\n        self._subscribers: tuple[SubscriberRegistration, ...] = ()\n        self._hooks: tuple[HookRegistration, ...] = ()\n\n    @property\n    def history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(self._history)\n\n    @property\n    def effective_history(self) -> tuple[ConversationMessage, ...]:\n        return tuple(\n            self._history\n            if self._effective_history is None\n            else self._effective_history\n        )\n\n    @property\n    def model(self) -> ModelSpec:\n        return self._model\n\n    @property\n    def tools(self) -> tuple[Tool, ...]:\n        return tuple(self._tools.values())\n\n    def configure_tools(self, tools: Sequence[Tool]) -> None:\n        """Replace the effective Tool set before a Session accepts work."""\n\n        registered: dict[str, Tool] = {}\n        for tool in tools:\n            if not isinstance(tool, Tool):\n                raise TypeError("tools must contain Tool values")\n            if tool.name in registered:\n                raise ValueError(f"duplicate Tool name: {tool.name!r}")\n            registered[tool.name] = tool\n        if registered and not self._model.supports_tools:\n            raise ValueError("configured ModelSpec does not support Tools")\n        self._tools = registered\n\n    def configure_subscribers(\n        self,\n        subscribers: Sequence[SubscriberRegistration],\n    ) -> None:\n        """Install high-level passive observers before accepting a Run."""\n\n        accepted = tuple(subscribers)\n        if any(not isinstance(item, SubscriberRegistration) for item in accepted):\n            raise TypeError("subscribers must contain SubscriberRegistration values")\n        self._subscribers = accepted\n\n    def configure_hooks(self, hooks: Sequence[HookRegistration]) -> None:\n        """Install the bounded ordered Hook set before accepting a Run."""\n\n        accepted = tuple(hooks)\n        if any(not isinstance(item, HookRegistration) for item in accepted):\n            raise TypeError("hooks must contain HookRegistration values")\n        self._hooks = accepted\n\n    async def _apply_hooks(\n        self,\n        point: HookPoint,\n        value: object,\n        snapshot: RunSnapshot | None = None,\n    ) -> object:\n        current = value\n        registrations = self._hooks if snapshot is None else snapshot.hooks\n        for registration in registrations:\n            if registration.point is not point:\n                continue\n            try:\n                replacement = await registration.callback(\n                    HookContext(point, current, snapshot)\n                )\n            except Exception as error:\n                raise HookExecutionError(\n                    registration.extension_name,\n                    point,\n                    type(error).__name__,\n                ) from error\n            if replacement is not None:\n                current = replacement\n        return current\n\n    async def apply_hooks(\n        self,\n        point: HookPoint,\n        value: object,\n        *,\n        snapshot: RunSnapshot | None = None,\n    ) -> object:\n        """Invoke one frozen-order Hook point for an AgentSession operation."""\n\n        return await self._apply_hooks(point, value, snapshot)\n\n    def capture_snapshot(\n        self,\n        *,\n        extension_identities: Sequence[str] = (),\n        compaction_policy: object | None = None,\n        prompt_hashes: Mapping[str, str] | None = None,\n        resource_hashes: Mapping[str, str] | None = None,\n    ) -> RunSnapshot:\n        return RunSnapshot.capture(\n            adapter=self._adapter,\n            model=self._model,\n            tools=self.tools,\n            tool_executor=self._tool_executor,\n            tool_output_budget=self._tool_output_budget,\n            retry_policy=self._retry_policy,\n            sleeper=self._sleeper,\n            run_guard=self._run_guard,\n            extension_identities=extension_identities,\n            subscribers=self._subscribers,\n            hooks=self._hooks,\n            generation_settings=self._generation_settings,\n            compaction_policy=compaction_policy,\n            prompt_hashes=prompt_hashes,\n            resource_hashes=resource_hashes,\n        )\n\n    @property\n    def run_guard(self) -> object | None:\n        return self._run_guard\n\n    def restore_history(\n        self,\n        history: Sequence[ConversationMessage],\n        *,\n        effective_history: Sequence[ConversationMessage] | None = None,\n    ) -> None:\n        """Seed a newly constructed Runtime from one settled Session branch."""\n\n        if self._history:\n            raise RuntimeError("Runtime history must be empty before restoration")\n        accepted = tuple(history)\n        to_model_messages(accepted)\n        self._history.extend(accepted)\n        if effective_history is not None:\n            effective = tuple(effective_history)\n            to_model_messages(effective)\n            self._effective_history = list(effective)\n\n    def install_compaction(self, checkpoint: CompactionCheckpoint) -> None:\n        """Replace only the model-facing prefix at a Settled Boundary."""\n\n        self._effective_history = [\n            checkpoint.summary.as_message(),\n            *checkpoint.retained_tail,\n        ]\n\n    def set_settlement_sink(self, sink: SettlementSink | None) -> None:\n        """Install the AgentSession-owned persistence barrier for the next Run."""\n\n        if sink is not None and not callable(sink):\n            raise TypeError("settlement sink must be callable")\n        self._settlement_sink = sink\n\n    def set_context_overflow_recovery(\n        self,\n        recovery: ContextOverflowRecovery | None,\n    ) -> None:\n        if recovery is not None and not callable(recovery):\n            raise TypeError("context overflow recovery must be callable")\n        self._context_overflow_recovery = recovery\n\n    def set_settled_turn_handler(\n        self,\n        handler: SettledTurnHandler | None,\n    ) -> None:\n        if handler is not None and not callable(handler):\n            raise TypeError("settled turn handler must be callable")\n        self._settled_turn_handler = handler\n\n    def _append_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        self._history.extend(accepted)\n        if self._effective_history is not None:\n            self._effective_history.extend(accepted)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def _append_unsettled(self, message: ConversationMessage) -> None:\n        self._history.append(message)\n        if self._effective_history is not None:\n            self._effective_history.append(message)\n\n    def _mark_settled(self, messages: Sequence[ConversationMessage]) -> None:\n        accepted = tuple(messages)\n        if accepted and self._settlement_sink is not None:\n            self._settlement_sink(accepted)\n\n    def start(\n        self,\n        messages: Sequence[AgentMessage],\n        *,\n        turn_input: TurnInput | None = None,\n        snapshot: RunSnapshot | None = None,\n    ) -> AgentRunHandle:\n        accepted = tuple(messages)\n        to_model_messages((*self.effective_history, *accepted))\n        self._append_settled(accepted)\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        cancellation = _CancellationState()\n        accepted_snapshot = snapshot or self.capture_snapshot()\n        if not isinstance(accepted_snapshot, RunSnapshot):\n            raise TypeError("snapshot must be a RunSnapshot")\n        task = asyncio.get_running_loop().create_task(\n            self._execute(events, cancellation, turn_input, accepted_snapshot)\n        )\n        handle = AgentRunHandle(task, events, cancellation, accepted_snapshot)\n        if (\n            isinstance(accepted_snapshot.run_guard, RunGuard)\n            and accepted_snapshot.run_guard.timeout_seconds is not None\n        ):\n            timer = asyncio.get_running_loop().call_later(\n                accepted_snapshot.run_guard.timeout_seconds,\n                handle._cancel_with,\n                TerminalStatus.TIMEOUT,\n            )\n            task.add_done_callback(lambda completed: timer.cancel())\n        return handle\n\n    async def run(self, messages: Sequence[AgentMessage]) -> AssistantOutcome:\n        return await self.start(messages).result()\n\n    async def generate_summary(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        focus: str | None = None,\n        snapshot: RunSnapshot | None = None,\n    ) -> SummaryGeneration:\n        """Run a retryable model operation without mutating conversation history."""\n\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("summary generation requires source history")\n        effective = snapshot or self.capture_snapshot()\n        instruction = (\n            "Summarize the following settled conversation as a durable context "\n            "checkpoint. Preserve decisions, constraints, unresolved work, and "\n            "facts needed to continue. Return summary text only."\n        )\n        if focus is not None:\n            instruction += f"\\nFocus requested by the caller: {focus}"\n        summary_prompt = AgentMessage.text(Role.SYSTEM, instruction)\n        attempt = 0\n        while True:\n            attempt += 1\n            request = ModelRequest(\n                messages=to_model_messages((summary_prompt, *accepted)),\n                model=effective.model,\n                operation=ModelOperation.COMPACTION,\n            )\n            text_parts: list[str] = []\n            usage: Usage | None = None\n            end: ModelEnd | None = None\n            schema_error: ModelError | None = None\n            try:\n                async for event in effective.adapter.stream(request):\n                    if end is not None:\n                        schema_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "Compaction stream emitted data after ModelEnd",\n                            False,\n                        )\n                        break\n                    if isinstance(event, TextDelta):\n                        text_parts.append(event.text)\n                    elif isinstance(event, UsageUpdate):\n                        usage = event.usage\n                    elif isinstance(event, ModelEnd):\n                        end = event\n                    else:\n                        schema_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "Compaction summary must contain text only",\n                            False,\n                        )\n                        break\n            except ModelAdapterError as failure:\n                delay = effective.retry_policy.delay_for(\n                    failure.error,\n                    attempt,\n                    retry_after_seconds=failure.error.retry_after_seconds,\n                )\n                if delay is None:\n                    raise\n                await effective.sleeper(delay)\n                continue\n            if schema_error is None and end is None:\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction stream did not end with ModelEnd",\n                    False,\n                )\n            if (\n                schema_error is None\n                and end is not None\n                and end.stop_reason is not StopReason.COMPLETE\n            ):\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction summary did not complete successfully",\n                    False,\n                )\n            text = "".join(text_parts)\n            if schema_error is None and not text.strip():\n                schema_error = ModelError(\n                    ModelErrorCode.SCHEMA,\n                    "Compaction summary cannot be empty",\n                    False,\n                )\n            if schema_error is not None:\n                raise ModelAdapterError(schema_error)\n            return SummaryGeneration(text, usage, attempt)\n\n    async def _execute(\n        self,\n        events: asyncio.Queue[RuntimeEvent | object],\n        cancellation: _CancellationState,\n        turn_input: TurnInput | None = None,\n        snapshot: RunSnapshot | None = None,\n    ) -> AssistantOutcome:\n        if snapshot is None:\n            snapshot = self.capture_snapshot()\n        tools = {tool.name: tool for tool in snapshot.tools}\n        request_history: list[ConversationMessage] = list(self.effective_history)\n        sequence = 0\n        total_attempts = 0\n        text_parts: list[str] = []\n        current_usage: Usage | None = None\n        run_usage: Usage | None = None\n        run_tool_results: list[ToolResult] = []\n        turns = 0\n        tool_calls = 0\n        started_at = asyncio.get_running_loop().time()\n        overflow_recovery_attempted = False\n\n        def append_settled(messages: Sequence[ConversationMessage]) -> None:\n            accepted = tuple(messages)\n            self._append_settled(accepted)\n            request_history.extend(accepted)\n\n        def append_unsettled(message: ConversationMessage) -> None:\n            self._append_unsettled(message)\n            request_history.append(message)\n\n        def guard_status() -> TerminalStatus | None:\n            if not isinstance(snapshot.run_guard, RunGuard):\n                return None\n            return snapshot.run_guard.reached(\n                turns=turns,\n                tool_calls=tool_calls,\n                usage=run_usage,\n                elapsed_seconds=asyncio.get_running_loop().time() - started_at,\n            )\n\n        async def emit(\n            type_: EventType,\n            *,\n            attempt: int | None = None,\n            model_event: ModelEvent | None = None,\n            error: ModelError | None = None,\n            retry_delay_seconds: float | None = None,\n            partial_text: str = "",\n            partial_usage: Usage | None = None,\n            tool_call_id: str | None = None,\n            tool_name: str | None = None,\n            tool_result: ToolResult | None = None,\n            operation: ModelOperation = ModelOperation.RUN,\n        ) -> None:\n            nonlocal sequence\n            sequence += 1\n            event = RuntimeEvent(\n                sequence=sequence,\n                type=type_,\n                attempt=attempt,\n                model_event=model_event,\n                error=error,\n                retry_delay_seconds=retry_delay_seconds,\n                partial_text=partial_text,\n                partial_usage=partial_usage,\n                tool_call_id=tool_call_id,\n                tool_name=tool_name,\n                tool_result=tool_result,\n                operation=operation,\n                run_id=snapshot.run_id,\n                snapshot_fingerprint=snapshot.fingerprint,\n            )\n            await events.put(event)\n            for registration in snapshot.subscribers:\n                try:\n                    await registration.callback(event)\n                except Exception as subscriber_error:\n                    sequence += 1\n                    await events.put(\n                        RuntimeEvent(\n                            sequence=sequence,\n                            type=EventType.SUBSCRIBER_FAILED,\n                            attempt=attempt,\n                            operation=operation,\n                            extension_name=registration.extension_name,\n                            diagnostic=type(subscriber_error).__name__,\n                            run_id=snapshot.run_id,\n                            snapshot_fingerprint=snapshot.fingerprint,\n                        )\n                    )\n\n        async def finish(outcome: AssistantOutcome) -> AssistantOutcome:\n            append_settled((outcome.message,))\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n\n        try:\n            await emit(EventType.AGENT_START)\n            try:\n                hooked_history = await self._apply_hooks(\n                    HookPoint.BEFORE_RUN,\n                    tuple(request_history),\n                    snapshot,\n                )\n                if isinstance(hooked_history, (str, bytes)) or not isinstance(\n                    hooked_history, Sequence\n                ):\n                    raise TypeError(\n                        "before_run Hook must return a message sequence or None"\n                    )\n                accepted_hooked_history = tuple(hooked_history)\n                to_model_messages(accepted_hooked_history)\n                request_history[:] = accepted_hooked_history\n            except (HookExecutionError, TypeError, ValueError) as hook_error:\n                error = ModelError(\n                    ModelErrorCode.HOOK_FAILED,\n                    str(hook_error),\n                    False,\n                )\n                return await finish(\n                    AssistantOutcome(\n                        AgentMessage.text(Role.ASSISTANT, ""),\n                        StopReason.ERROR,\n                        run_usage,\n                        error,\n                        1,\n                        tuple(run_tool_results),\n                        TerminalStatus.MODEL_ERROR,\n                    )\n                )\n            while True:\n                turn_attempt = 0\n                while True:\n                    turn_attempt += 1\n                    total_attempts += 1\n                    attempt = total_attempts\n                    request = ModelRequest(\n                        to_model_messages(request_history),\n                        snapshot.model,\n                        tuple(tool.definition() for tool in tools.values()),\n                    )\n                    try:\n                        hooked_request = await self._apply_hooks(\n                            HookPoint.BEFORE_MODEL_REQUEST, request, snapshot\n                        )\n                        if not isinstance(hooked_request, ModelRequest):\n                            raise TypeError(\n                                "before_model_request Hook must return ModelRequest or None"\n                            )\n                        request = hooked_request\n                    except (HookExecutionError, TypeError) as hook_error:\n                        error = ModelError(\n                            ModelErrorCode.HOOK_FAILED,\n                            str(hook_error),\n                            False,\n                        )\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, ""),\n                                StopReason.ERROR,\n                                run_usage,\n                                error,\n                                max(total_attempts, 1),\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n                    await emit(EventType.MODEL_ATTEMPT_START, attempt=attempt)\n                    text_parts = []\n                    tool_drafts: dict[int, dict[str, str]] = {}\n                    current_usage = None\n                    end: ModelEnd | None = None\n                    schema_error: ModelError | None = None\n                    try:\n                        async for event in snapshot.adapter.stream(request):\n                            await emit(\n                                EventType.MODEL_EVENT,\n                                attempt=attempt,\n                                model_event=event,\n                            )\n                            if end is not None:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted data after ModelEnd",\n                                    False,\n                                )\n                                break\n                            if isinstance(event, TextDelta):\n                                text_parts.append(event.text)\n                            elif isinstance(event, ToolCallDelta):\n                                if event.index < 0:\n                                    schema_error = ModelError(\n                                        ModelErrorCode.SCHEMA,\n                                        "model stream emitted an invalid Tool Call index",\n                                        False,\n                                    )\n                                    break\n                                draft = tool_drafts.setdefault(\n                                    event.index,\n                                    {"id": "", "name": "", "arguments": ""},\n                                )\n                                draft["id"] += event.id\n                                draft["name"] += event.name\n                                draft["arguments"] += event.arguments_delta\n                            elif isinstance(event, UsageUpdate):\n                                current_usage = event.usage\n                            elif isinstance(event, ModelEnd):\n                                end = event\n                            else:\n                                schema_error = ModelError(\n                                    ModelErrorCode.SCHEMA,\n                                    "model stream emitted an unsupported event",\n                                    False,\n                                )\n                                break\n                    except ModelAdapterError as failure:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=failure.error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        if (\n                            failure.error.code is ModelErrorCode.CONTEXT_OVERFLOW\n                            and not overflow_recovery_attempted\n                            and self._context_overflow_recovery is not None\n                        ):\n                            overflow_recovery_attempted = True\n                            await emit(\n                                EventType.COMPACTION_START,\n                                attempt=attempt,\n                                error=failure.error,\n                                operation=ModelOperation.COMPACTION,\n                            )\n                            try:\n                                recovered = await self._context_overflow_recovery()\n                            except Exception as recovery_error:\n                                recovered = False\n                                failure = ModelAdapterError(\n                                    ModelError(\n                                        ModelErrorCode.COMPACTION_FAILED,\n                                        "context overflow recovery Compaction failed: "\n                                        f"{type(recovery_error).__name__}",\n                                        False,\n                                    )\n                                )\n                            await emit(\n                                (\n                                    EventType.COMPACTION_END\n                                    if recovered\n                                    else EventType.COMPACTION_FAILED\n                                ),\n                                attempt=attempt,\n                                error=None if recovered else failure.error,\n                                operation=ModelOperation.COMPACTION,\n                            )\n                            if recovered:\n                                request_history[:] = self.effective_history\n                                text_parts = []\n                                current_usage = None\n                                continue\n                        delay = snapshot.retry_policy.delay_for(\n                            failure.error,\n                            turn_attempt,\n                            retry_after_seconds=failure.error.retry_after_seconds,\n                        )\n                        if delay is not None:\n                            await emit(\n                                EventType.RETRY_SCHEDULED,\n                                attempt=attempt,\n                                error=failure.error,\n                                retry_delay_seconds=delay,\n                                partial_text=partial_text,\n                                partial_usage=current_usage,\n                            )\n                            text_parts = []\n                            current_usage = None\n                            await snapshot.sleeper(delay)\n                            continue\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                failure.error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n\n                    stream_error = schema_error\n                    if stream_error is None and end is None:\n                        stream_error = ModelError(\n                            ModelErrorCode.SCHEMA,\n                            "model stream violated the provider-neutral event contract",\n                            False,\n                        )\n                    blocks: list[ContentBlock] = []\n                    if stream_error is None:\n                        try:\n                            if text_parts:\n                                blocks.append(TextContent("".join(text_parts)))\n                            for index in sorted(tool_drafts):\n                                blocks.append(ToolCallContent(**tool_drafts[index]))\n                        except (TypeError, ValueError):\n                            stream_error = ModelError(\n                                ModelErrorCode.SCHEMA,\n                                "model stream emitted an incomplete Tool Call",\n                                False,\n                            )\n                    if stream_error is not None:\n                        partial_text = "".join(text_parts)\n                        await emit(\n                            EventType.MODEL_ATTEMPT_FAILED,\n                            attempt=attempt,\n                            error=stream_error,\n                            partial_text=partial_text,\n                            partial_usage=current_usage,\n                        )\n                        terminal_usage = _add_usage(run_usage, current_usage)\n                        return await finish(\n                            AssistantOutcome(\n                                AgentMessage.text(Role.ASSISTANT, partial_text),\n                                StopReason.ERROR,\n                                terminal_usage,\n                                stream_error,\n                                total_attempts,\n                                tuple(run_tool_results),\n                                TerminalStatus.MODEL_ERROR,\n                            )\n                        )\n                    assert end is not None\n                    break\n\n                run_usage = _add_usage(run_usage, current_usage)\n                turns += 1\n                assistant = AgentMessage(Role.ASSISTANT, tuple(blocks))\n                calls = tuple(\n                    block\n                    for block in assistant.content\n                    if isinstance(block, ToolCallContent)\n                )\n                if calls:\n                    append_unsettled(assistant)\n                else:\n                    append_settled((assistant,))\n                await emit(EventType.MESSAGE_END, attempt=total_attempts)\n                if not calls:\n                    if self._settled_turn_handler is not None:\n                        previous_effective = self.effective_history\n                        await self._settled_turn_handler()\n                        if self.effective_history != previous_effective:\n                            request_history[:] = self.effective_history\n                    if turn_input is not None and turn_input.pending():\n                        reached = guard_status()\n                        if reached is not None:\n                            await emit(EventType.AGENT_END, attempt=total_attempts)\n                            return AssistantOutcome(\n                                assistant,\n                                StopReason.ABORTED,\n                                run_usage,\n                                attempts=total_attempts,\n                                tool_results=tuple(run_tool_results),\n                                status=reached,\n                            )\n                        steering = tuple(turn_input.take())\n                        to_model_messages(steering)\n                        append_settled(steering)\n                        continue\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n\n                await emit(EventType.TOOL_BATCH_START, attempt=total_attempts)\n                prepared: dict[int, PreparedToolCall] = {}\n                results: dict[int, ToolResult] = {}\n                for index, call in enumerate(calls):\n                    tool = tools.get(call.name)\n                    if tool is None:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.UNKNOWN_TOOL, call.name\n                        )\n                        continue\n                    try:\n                        parsed = json.loads(call.arguments)\n                    except (json.JSONDecodeError, TypeError):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_JSON, call.name\n                        )\n                        continue\n                    try:\n                        Draft202012Validator(tool.input_schema).validate(parsed)\n                    except ValidationError:\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    if not isinstance(parsed, dict):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.INVALID_ARGUMENTS, call.name\n                        )\n                        continue\n                    candidate = PreparedToolCall(call, tool, parsed)\n                    try:\n                        hooked_call = await self._apply_hooks(\n                            HookPoint.BEFORE_TOOL_CALL,\n                            candidate,\n                            snapshot,\n                        )\n                        if not isinstance(hooked_call, PreparedToolCall):\n                            raise TypeError(\n                                "before_tool_call Hook must return PreparedToolCall or None"\n                            )\n                    except (HookExecutionError, TypeError):\n                        results[index] = ToolResult.error(\n                            ToolErrorCode.HOOK_FAILED,\n                            call.name,\n                        )\n                        continue\n                    prepared[index] = hooked_call\n\n                async def execute_one(index: int, call: PreparedToolCall) -> None:\n                    await emit(\n                        EventType.TOOL_CALL_START,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                    )\n                    try:\n                        result = await snapshot.tool_executor.execute(call)\n                        if not isinstance(result, ToolResult):\n                            raise TypeError("ToolExecutor returned an invalid result")\n                    except Exception:\n                        result = ToolResult.error(\n                            ToolErrorCode.EXECUTION_FAILED, call.call.name\n                        )\n                    try:\n                        hooked_result = await self._apply_hooks(\n                            HookPoint.AFTER_TOOL_CALL,\n                            result,\n                            snapshot,\n                        )\n                        if not isinstance(hooked_result, ToolResult):\n                            raise TypeError(\n                                "after_tool_call Hook must return ToolResult or None"\n                            )\n                        result = hooked_result\n                    except (HookExecutionError, TypeError):\n                        result = ToolResult.error(\n                            ToolErrorCode.HOOK_FAILED,\n                            call.call.name,\n                        )\n                    result = bound_tool_result(\n                        result,\n                        snapshot.tool_output_budget,\n                        call.tool.output_direction,\n                    )\n                    results[index] = result\n                    await emit(\n                        EventType.TOOL_CALL_END,\n                        attempt=total_attempts,\n                        tool_call_id=call.call.id,\n                        tool_name=call.call.name,\n                        tool_result=result,\n                    )\n\n                try:\n                    if any(call.tool.sequential for call in prepared.values()):\n                        for index, prepared_call in prepared.items():\n                            await execute_one(index, prepared_call)\n                    else:\n                        tasks = {\n                            index: asyncio.create_task(\n                                execute_one(index, prepared_call)\n                            )\n                            for index, prepared_call in prepared.items()\n                        }\n                        try:\n                            await asyncio.gather(*tasks.values())\n                        except asyncio.CancelledError:\n                            for task in tasks.values():\n                                if not task.done():\n                                    task.cancel()\n                            await asyncio.gather(\n                                *tasks.values(), return_exceptions=True\n                            )\n                            raise\n                except asyncio.CancelledError:\n                    for index, prepared_call in prepared.items():\n                        if index not in results:\n                            cancelled_result = ToolResult.error(\n                                ToolErrorCode.CANCELLED,\n                                prepared_call.call.name,\n                            )\n                            results[index] = cancelled_result\n                            await emit(\n                                EventType.TOOL_CALL_END,\n                                attempt=total_attempts,\n                                tool_call_id=prepared_call.call.id,\n                                tool_name=prepared_call.call.name,\n                                tool_result=cancelled_result,\n                            )\n                    for index, call in enumerate(calls):\n                        result = results[index]\n                        run_tool_results.append(result)\n                        append_unsettled(\n                            ToolResultMessage(call.id, call.name, result)\n                        )\n                    self._mark_settled(self._history[-(len(calls) + 1) :])\n                    await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                    raise\n\n                batch_results: list[ToolResult] = []\n                for index, call in enumerate(calls):\n                    result = results[index]\n                    batch_results.append(result)\n                    run_tool_results.append(result)\n                    append_unsettled(\n                        ToolResultMessage(call.id, call.name, result)\n                    )\n                self._mark_settled(self._history[-(len(calls) + 1) :])\n                tool_calls += len(calls)\n                await emit(EventType.TOOL_BATCH_END, attempt=total_attempts)\n                if self._settled_turn_handler is not None:\n                    previous_effective = self.effective_history\n                    await self._settled_turn_handler()\n                    if self.effective_history != previous_effective:\n                        request_history[:] = self.effective_history\n                if batch_results and all(result.terminate for result in batch_results):\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        end.stop_reason,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                    )\n                reached = guard_status()\n                if reached is not None:\n                    await emit(EventType.AGENT_END, attempt=total_attempts)\n                    return AssistantOutcome(\n                        assistant,\n                        StopReason.ABORTED,\n                        run_usage,\n                        attempts=total_attempts,\n                        tool_results=tuple(run_tool_results),\n                        status=reached,\n                    )\n                if turn_input is not None and turn_input.pending():\n                    steering = tuple(turn_input.take())\n                    to_model_messages(steering)\n                    append_settled(steering)\n        except asyncio.CancelledError:\n            message = AgentMessage.text(Role.ASSISTANT, "".join(text_parts))\n            outcome = AssistantOutcome(\n                message,\n                StopReason.ABORTED,\n                _add_usage(run_usage, current_usage),\n                attempts=max(total_attempts, 1),\n                tool_results=tuple(run_tool_results),\n                status=cancellation.status,\n            )\n            append_settled((message,))\n            await emit(\n                EventType.RUN_CANCELLED,\n                attempt=max(total_attempts, 1),\n                partial_text="".join(text_parts),\n                partial_usage=current_usage,\n            )\n            await emit(EventType.MESSAGE_END, attempt=max(total_attempts, 1))\n            await emit(EventType.AGENT_END, attempt=max(total_attempts, 1))\n            return outcome\n        finally:\n            await events.put(_EVENTS_DONE)\n'


In [ ]:
PERSISTENCE_SOURCE = '"""Tree-structured Session persistence at settled boundaries."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom enum import Enum\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom typing import BinaryIO, Protocol, TypeAlias\nfrom uuid import uuid4\n\nfrom .compaction import (\n    CompactionCheckpoint,\n    CompactionTrigger,\n    StructuredSummary,\n)\nfrom .extensions import CustomEntry\nfrom .model import AgentMessage, Role, TextContent, ToolCallContent, Usage\nfrom .tools import (\n    CompleteOutputKind,\n    CompleteOutputReference,\n    ToolErrorCode,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\n\n\nConversationMessage: TypeAlias = AgentMessage | ToolResultMessage\nIdFactory: TypeAlias = Callable[[], str]\n\n\n@dataclass(frozen=True, slots=True)\nclass SchemaVersion:\n    major: int\n    minor: int = 0\n\n\nSESSION_SCHEMA_VERSION = SchemaVersion(1, 2)\n_SESSION_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]{0,127}\\Z")\n\n\nclass RecoveryCode(str, Enum):\n    INCOMPLETE_FINAL_RECORD = "incomplete_final_record"\n\n\n@dataclass(frozen=True, slots=True)\nclass RecoveryWarning:\n    code: RecoveryCode\n    message: str\n    line_number: int\n\n\nclass SessionBusyError(RuntimeError):\n    """Structured failure raised when a Session already owns its writer lease."""\n\n    code = "session_busy"\n\n    def __init__(self, session_id: str, message: str | None = None) -> None:\n        self.session_id = session_id\n        super().__init__(\n            message or f"Session {session_id!r} already has an active writer"\n        )\n\n\nclass UnsupportedSchemaVersionError(ValueError):\n    def __init__(self, found_major: int) -> None:\n        self.artifact = "Session"\n        self.found_major = found_major\n        self.supported_major = SESSION_SCHEMA_VERSION.major\n        super().__init__(\n            f"Session schema major {found_major} is unsupported; "\n            f"this reader supports major {self.supported_major}. "\n            "Preserve the original and migrate it with migrate_session_file()."\n        )\n\n\ndef _validate_record_version(record: object, line_number: int) -> Mapping[str, object]:\n    if not isinstance(record, dict):\n        raise ValueError(f"Session record at line {line_number} must be an object")\n    if record.get("schema") != "agent_harness.session":\n        raise ValueError(f"invalid Session schema at line {line_number}")\n    raw_version = record.get("schema_version")\n    if not isinstance(raw_version, dict):\n        raise ValueError(f"invalid Session schema version at line {line_number}")\n    major = raw_version.get("major")\n    minor = raw_version.get("minor")\n    if (\n        isinstance(major, bool)\n        or not isinstance(major, int)\n        or isinstance(minor, bool)\n        or not isinstance(minor, int)\n        or major < 1\n        or minor < 0\n    ):\n        raise ValueError(f"invalid Session schema version at line {line_number}")\n    if major != SESSION_SCHEMA_VERSION.major:\n        raise UnsupportedSchemaVersionError(major)\n    return record\n\n\ndef _session_id(value: str) -> str:\n    if not _SESSION_ID.fullmatch(value):\n        raise ValueError("Session id must be a safe local identifier")\n    return value\n\n\ndef _version_record() -> dict[str, int]:\n    return {\n        "major": SESSION_SCHEMA_VERSION.major,\n        "minor": SESSION_SCHEMA_VERSION.minor,\n    }\n\n\ndef _encode_message(message: ConversationMessage) -> dict[str, object]:\n    if isinstance(message, AgentMessage):\n        content: list[dict[str, object]] = []\n        for block in message.content:\n            if isinstance(block, TextContent):\n                content.append({"type": "text", "text": block.text})\n            elif isinstance(block, ToolCallContent):\n                content.append(\n                    {\n                        "type": "tool_call",\n                        "id": block.id,\n                        "name": block.name,\n                        "arguments": block.arguments,\n                    }\n                )\n            else:\n                raise TypeError(f"unsupported Content Block: {type(block).__name__}")\n        return {\n            "kind": "agent_message",\n            "role": message.role.value,\n            "content": content,\n        }\n    if isinstance(message, ToolResultMessage):\n        result = message.result\n        truncation = (\n            None\n            if result.truncation is None\n            else {\n                "original_bytes": result.truncation.original_bytes,\n                "original_lines": result.truncation.original_lines,\n                "retained_start_byte": result.truncation.retained_start_byte,\n                "retained_end_byte": result.truncation.retained_end_byte,\n                "retained_start_line": result.truncation.retained_start_line,\n                "retained_end_line": result.truncation.retained_end_line,\n                "direction": result.truncation.direction.value,\n            }\n        )\n        complete_output = (\n            None\n            if result.complete_output is None\n            else {\n                "kind": result.complete_output.kind.value,\n                "reference": result.complete_output.reference,\n                "reason": result.complete_output.reason,\n            }\n        )\n        return {\n            "kind": "tool_result",\n            "tool_call_id": message.tool_call_id,\n            "tool_name": message.tool_name,\n            "result": {\n                "content": result.content,\n                "metadata": dict(result.metadata),\n                "terminate": result.terminate,\n                "is_error": result.is_error,\n                "error_code": (\n                    None if result.error_code is None else result.error_code.value\n                ),\n                "truncation": truncation,\n                "complete_output": complete_output,\n            },\n        }\n    raise TypeError(f"unsupported Session message: {type(message).__name__}")\n\n\ndef _decode_message(record: Mapping[str, object]) -> ConversationMessage:\n    if record.get("kind") == "agent_message":\n        role = Role(str(record["role"]))\n        raw_content = record.get("content")\n        if not isinstance(raw_content, list):\n            raise ValueError("Session AgentMessage content must be a list")\n        blocks: list[TextContent | ToolCallContent] = []\n        for raw_block in raw_content:\n            if not isinstance(raw_block, dict):\n                raise ValueError("Session Content Block must be an object")\n            if raw_block.get("type") == "text":\n                blocks.append(TextContent(str(raw_block["text"])))\n            elif raw_block.get("type") == "tool_call":\n                blocks.append(\n                    ToolCallContent(\n                        str(raw_block["id"]),\n                        str(raw_block["name"]),\n                        str(raw_block["arguments"]),\n                    )\n                )\n            else:\n                raise ValueError("unsupported Session Content Block type")\n        return AgentMessage(role, tuple(blocks))\n    if record.get("kind") == "tool_result":\n        raw_result = record.get("result")\n        if not isinstance(raw_result, dict):\n            raise ValueError("Session ToolResult must be an object")\n        raw_truncation = raw_result.get("truncation")\n        truncation = None\n        if isinstance(raw_truncation, dict):\n            truncation = TruncationNotice(\n                int(raw_truncation["original_bytes"]),\n                int(raw_truncation["original_lines"]),\n                int(raw_truncation["retained_start_byte"]),\n                int(raw_truncation["retained_end_byte"]),\n                int(raw_truncation["retained_start_line"]),\n                int(raw_truncation["retained_end_line"]),\n                TruncationDirection(str(raw_truncation["direction"])),\n            )\n        raw_complete = raw_result.get("complete_output")\n        complete_output = None\n        if isinstance(raw_complete, dict):\n            complete_output = CompleteOutputReference(\n                CompleteOutputKind(str(raw_complete["kind"])),\n                None\n                if raw_complete.get("reference") is None\n                else str(raw_complete["reference"]),\n                None\n                if raw_complete.get("reason") is None\n                else str(raw_complete["reason"]),\n            )\n        raw_metadata = raw_result.get("metadata", {})\n        if not isinstance(raw_metadata, dict):\n            raise ValueError("Session ToolResult metadata must be an object")\n        raw_error_code = raw_result.get("error_code")\n        result = ToolResult(\n            str(raw_result["content"]),\n            metadata=raw_metadata,\n            terminate=bool(raw_result.get("terminate", False)),\n            is_error=bool(raw_result.get("is_error", False)),\n            error_code=(\n                None\n                if raw_error_code is None\n                else ToolErrorCode(str(raw_error_code))\n            ),\n            truncation=truncation,\n            complete_output=complete_output,\n        )\n        return ToolResultMessage(\n            str(record["tool_call_id"]), str(record["tool_name"]), result\n        )\n    raise ValueError("unsupported Session message kind")\n\n\ndef _encode_compaction(checkpoint: CompactionCheckpoint) -> dict[str, object]:\n    usage = checkpoint.summary_usage\n    return {\n        "trigger": checkpoint.trigger.value,\n        "summary": {\n            "schema_version": checkpoint.summary.schema_version,\n            "text": checkpoint.summary.text,\n            "focus": checkpoint.summary.focus,\n        },\n        "tokens_before": checkpoint.tokens_before,\n        "summary_usage": (\n            None\n            if usage is None\n            else {\n                "input_tokens": usage.input_tokens,\n                "output_tokens": usage.output_tokens,\n                "total_tokens": usage.total_tokens,\n                "estimated": usage.estimated,\n            }\n        ),\n        "retained_tail": [\n            _encode_message(message) for message in checkpoint.retained_tail\n        ],\n    }\n\n\ndef _decode_compaction(record: Mapping[str, object]) -> CompactionCheckpoint:\n    raw_summary = record.get("summary")\n    raw_tail = record.get("retained_tail")\n    if not isinstance(raw_summary, dict) or not isinstance(raw_tail, list):\n        raise ValueError("invalid Compaction checkpoint")\n    raw_usage = record.get("summary_usage")\n    usage = None\n    if isinstance(raw_usage, dict):\n        usage = Usage(\n            int(raw_usage["input_tokens"]),\n            int(raw_usage["output_tokens"]),\n            int(raw_usage["total_tokens"]),\n            bool(raw_usage.get("estimated", False)),\n        )\n    return CompactionCheckpoint(\n        trigger=CompactionTrigger(str(record["trigger"])),\n        summary=StructuredSummary(\n            text=str(raw_summary["text"]),\n            focus=(\n                None\n                if raw_summary.get("focus") is None\n                else str(raw_summary["focus"])\n            ),\n            schema_version=int(raw_summary.get("schema_version", 1)),\n        ),\n        tokens_before=int(str(record["tokens_before"])),\n        summary_usage=usage,\n        retained_tail=tuple(_decode_message(message) for message in raw_tail),\n    )\n\n\ndef _encode_custom_entry(entry: CustomEntry) -> dict[str, object]:\n    return {\n        "namespace": entry.namespace,\n        "version": entry.version,\n        "payload": dict(entry.payload),\n    }\n\n\ndef _decode_custom_entry(record: Mapping[str, object]) -> CustomEntry:\n    payload = record.get("payload")\n    version = record.get("version")\n    if not isinstance(payload, dict):\n        raise ValueError("Custom Entry payload must be an object")\n    if isinstance(version, bool) or not isinstance(version, int):\n        raise ValueError("Custom Entry version must be an integer")\n    return CustomEntry(\n        str(record["namespace"]),\n        version,\n        payload,\n    )\n\n\nclass SessionWriter(Protocol):\n    session_id: str\n\n    def __enter__(self) -> "SessionWriter": ...\n\n    def __exit__(self, *exc_info: object) -> None: ...\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> "SessionEntry": ...\n\n    def append_compaction(\n        self,\n        checkpoint: CompactionCheckpoint,\n        *,\n        parent_id: str | None = None,\n    ) -> "SessionEntry": ...\n\n    def append_custom(\n        self,\n        custom: CustomEntry,\n        *,\n        parent_id: str | None = None,\n    ) -> "SessionEntry": ...\n\n\nclass SessionStore(Protocol):\n    def create(self, session_id: str | None = None) -> "SessionState": ...\n\n    def read(self, session_id: str) -> "SessionState": ...\n\n    def writer(self, session_id: str) -> SessionWriter: ...\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionEntry:\n    entry_id: str\n    parent_id: str | None\n    messages: tuple[ConversationMessage, ...] = ()\n    compaction: CompactionCheckpoint | None = None\n    custom: CustomEntry | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionState:\n    session_id: str\n    entries: tuple[SessionEntry, ...] = ()\n    recovery_warning: RecoveryWarning | None = None\n\n    def __post_init__(self) -> None:\n        _session_id(self.session_id)\n        earlier: set[str] = set()\n        for entry in self.entries:\n            if not _SESSION_ID.fullmatch(entry.entry_id):\n                raise ValueError("Session entry id must be a safe local identifier")\n            if entry.entry_id in earlier:\n                raise ValueError(f"duplicate Session entry id: {entry.entry_id}")\n            if entry.parent_id is not None and entry.parent_id not in earlier:\n                raise ValueError(\n                    "Session entry parent must reference an earlier Session entry"\n                )\n            kinds = (\n                int(bool(entry.messages))\n                + int(entry.compaction is not None)\n                + int(entry.custom is not None)\n            )\n            if kinds != 1:\n                raise ValueError(\n                    "a Session entry requires messages, Compaction, or Custom Entry"\n                )\n            earlier.add(entry.entry_id)\n\n    @property\n    def active_leaf_id(self) -> str | None:\n        return self.entries[-1].entry_id if self.entries else None\n\n    def history(self, leaf_id: str | None = None) -> tuple[ConversationMessage, ...]:\n        if not self.entries:\n            if leaf_id is not None:\n                raise KeyError(f"unknown Session entry: {leaf_id}")\n            return ()\n        by_id = {entry.entry_id: entry for entry in self.entries}\n        cursor = self.active_leaf_id if leaf_id is None else leaf_id\n        path: list[SessionEntry] = []\n        while cursor is not None:\n            try:\n                entry = by_id[cursor]\n            except KeyError:\n                raise KeyError(f"unknown Session entry: {cursor}") from None\n            path.append(entry)\n            cursor = entry.parent_id\n        return tuple(\n            message for entry in reversed(path) for message in entry.messages\n        )\n\n    def compactions(\n        self, leaf_id: str | None = None\n    ) -> tuple[CompactionCheckpoint, ...]:\n        return tuple(\n            entry.compaction\n            for entry in self._path(leaf_id)\n            if entry.compaction is not None\n        )\n\n    def custom_entries(self, leaf_id: str | None = None) -> tuple[CustomEntry, ...]:\n        return tuple(\n            entry.custom\n            for entry in self._path(leaf_id)\n            if entry.custom is not None\n        )\n\n    def effective_history(\n        self, leaf_id: str | None = None\n    ) -> tuple[ConversationMessage, ...]:\n        path = self._path(leaf_id)\n        latest = next(\n            (\n                index\n                for index in range(len(path) - 1, -1, -1)\n                if path[index].compaction is not None\n            ),\n            None,\n        )\n        if latest is None:\n            return tuple(message for entry in path for message in entry.messages)\n        checkpoint = path[latest].compaction\n        assert checkpoint is not None\n        return (\n            checkpoint.summary.as_message(),\n            *checkpoint.retained_tail,\n            *(\n                message\n                for entry in path[latest + 1 :]\n                for message in entry.messages\n            ),\n        )\n\n    def _path(self, leaf_id: str | None = None) -> tuple[SessionEntry, ...]:\n        if not self.entries:\n            if leaf_id is not None:\n                raise KeyError(f"unknown Session entry: {leaf_id}")\n            return ()\n        by_id = {entry.entry_id: entry for entry in self.entries}\n        cursor = self.active_leaf_id if leaf_id is None else leaf_id\n        path: list[SessionEntry] = []\n        while cursor is not None:\n            try:\n                entry = by_id[cursor]\n            except KeyError:\n                raise KeyError(f"unknown Session entry: {cursor}") from None\n            path.append(entry)\n            cursor = entry.parent_id\n        return tuple(reversed(path))\n\n\n@dataclass(frozen=True, slots=True)\nclass MigrationResult:\n    source: Path\n    destination: Path\n    session_id: str\n    entries: int\n\n\nclass MemorySessionWriter:\n    def __init__(self, store: "MemorySessionStore", session_id: str) -> None:\n        self._store = store\n        self.session_id = session_id\n\n    def __enter__(self) -> "MemorySessionWriter":\n        return self\n\n    def __exit__(self, *exc_info: object) -> None:\n        return None\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("a settled Session entry requires messages")\n        entry = SessionEntry(self._store._id_factory(), parent, accepted)\n        self._store._sessions[self.session_id] = SessionState(\n            self.session_id, (*state.entries, entry)\n        )\n        return entry\n\n    def append_compaction(\n        self,\n        checkpoint: CompactionCheckpoint,\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        entry = SessionEntry(\n            self._store._id_factory(),\n            parent,\n            compaction=checkpoint,\n        )\n        self._store._sessions[self.session_id] = SessionState(\n            self.session_id, (*state.entries, entry)\n        )\n        return entry\n\n    def append_custom(\n        self,\n        custom: CustomEntry,\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        if not isinstance(custom, CustomEntry):\n            raise TypeError("custom must be a CustomEntry")\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        entry = SessionEntry(\n            self._store._id_factory(),\n            parent,\n            custom=custom,\n        )\n        self._store._sessions[self.session_id] = SessionState(\n            self.session_id, (*state.entries, entry)\n        )\n        return entry\n\n\nclass MemorySessionStore:\n    """In-process SessionStore adapter with the durable tree contract."""\n\n    def __init__(self, *, id_factory: IdFactory | None = None) -> None:\n        self._id_factory = id_factory or (lambda: uuid4().hex)\n        self._sessions: dict[str, SessionState] = {}\n\n    def create(self, session_id: str | None = None) -> SessionState:\n        accepted = _session_id(session_id or self._id_factory())\n        if accepted in self._sessions:\n            raise ValueError(f"Session already exists: {accepted}")\n        state = SessionState(accepted)\n        self._sessions[accepted] = state\n        return state\n\n    def read(self, session_id: str) -> SessionState:\n        try:\n            return self._sessions[session_id]\n        except KeyError:\n            raise KeyError(f"unknown Session: {session_id}") from None\n\n    def writer(self, session_id: str) -> MemorySessionWriter:\n        self.read(session_id)\n        return MemorySessionWriter(self, session_id)\n\n\nclass JSONLSessionWriter:\n    def __init__(self, store: "JSONLSessionStore", session_id: str) -> None:\n        self._store = store\n        self.session_id = session_id\n        self._lock_stream: BinaryIO | None = None\n\n    def __enter__(self) -> "JSONLSessionWriter":\n        if self._lock_stream is not None:\n            raise RuntimeError("Session writer lease is already active")\n        self._lock_stream = self._store._acquire_lock(self.session_id)\n        try:\n            state = self._store.read(self.session_id)\n            if state.recovery_warning is not None:\n                self._store._discard_uncommitted_tail(self.session_id)\n        except BaseException:\n            self._store._release_lock(self._lock_stream)\n            self._lock_stream = None\n            raise\n        return self\n\n    def __exit__(self, *exc_info: object) -> None:\n        if self._lock_stream is not None:\n            self._store._release_lock(self._lock_stream)\n            self._lock_stream = None\n\n    def append(\n        self,\n        messages: Sequence[ConversationMessage],\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        if self._lock_stream is None:\n            raise RuntimeError("Session writer lease is not active")\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        accepted = tuple(messages)\n        if not accepted:\n            raise ValueError("a settled Session entry requires messages")\n        entry = SessionEntry(self._store._id_factory(), parent, accepted)\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "settlement",\n            "session_id": self.session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n            "messages": [_encode_message(message) for message in accepted],\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        with self._store.path_for(self.session_id).open("a", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        return entry\n\n    def append_compaction(\n        self,\n        checkpoint: CompactionCheckpoint,\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        if self._lock_stream is None:\n            raise RuntimeError("Session writer lease is not active")\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        entry = SessionEntry(\n            self._store._id_factory(),\n            parent,\n            compaction=checkpoint,\n        )\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "compaction",\n            "session_id": self.session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n            "checkpoint": _encode_compaction(checkpoint),\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        with self._store.path_for(self.session_id).open("a", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        return entry\n\n    def append_custom(\n        self,\n        custom: CustomEntry,\n        *,\n        parent_id: str | None = None,\n    ) -> SessionEntry:\n        if self._lock_stream is None:\n            raise RuntimeError("Session writer lease is not active")\n        if not isinstance(custom, CustomEntry):\n            raise TypeError("custom must be a CustomEntry")\n        state = self._store.read(self.session_id)\n        parent = state.active_leaf_id if parent_id is None else parent_id\n        if parent is not None and parent not in {\n            entry.entry_id for entry in state.entries\n        }:\n            raise KeyError(f"unknown Session entry: {parent}")\n        entry = SessionEntry(\n            self._store._id_factory(),\n            parent,\n            custom=custom,\n        )\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "custom",\n            "session_id": self.session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n            "custom": _encode_custom_entry(custom),\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        with self._store.path_for(self.session_id).open("a", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        return entry\n\n\nclass JSONLSessionStore:\n    """Transparent file-per-Session JSONL adapter."""\n\n    def __init__(self, root: str | Path, *, id_factory: IdFactory | None = None) -> None:\n        self.root = Path(root)\n        self.sessions_directory = self.root / "sessions"\n        self._id_factory = id_factory or (lambda: uuid4().hex)\n\n    def path_for(self, session_id: str) -> Path:\n        return self.sessions_directory / f"{_session_id(session_id)}.jsonl"\n\n    def _lock_path(self, session_id: str) -> Path:\n        return self.sessions_directory / ".locks" / f"{_session_id(session_id)}.lock"\n\n    def _discard_uncommitted_tail(self, session_id: str) -> None:\n        path = self.path_for(session_id)\n        content = path.read_bytes()\n        committed_end = content.rfind(b"\\n")\n        if committed_end < 0:\n            raise ValueError("Session file has no committed header")\n        with path.open("r+b") as stream:\n            stream.truncate(committed_end + 1)\n            stream.flush()\n            os.fsync(stream.fileno())\n\n    def _acquire_lock(self, session_id: str) -> BinaryIO:\n        path = self._lock_path(session_id)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        stream = path.open("a+b")\n        try:\n            if os.name == "nt":\n                import msvcrt\n\n                stream.seek(0, os.SEEK_END)\n                if stream.tell() == 0:\n                    stream.write(b"\\0")\n                    stream.flush()\n                stream.seek(0)\n                msvcrt.locking(  # type: ignore[attr-defined]\n                    stream.fileno(), msvcrt.LK_NBLCK, 1  # type: ignore[attr-defined]\n                )\n            else:\n                import fcntl\n\n                fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except OSError:\n            stream.close()\n            raise SessionBusyError(session_id) from None\n        return stream\n\n    @staticmethod\n    def _release_lock(stream: BinaryIO) -> None:\n        try:\n            if os.name == "nt":\n                import msvcrt\n\n                stream.seek(0)\n                msvcrt.locking(  # type: ignore[attr-defined]\n                    stream.fileno(), msvcrt.LK_UNLCK, 1  # type: ignore[attr-defined]\n                )\n            else:\n                import fcntl\n\n                fcntl.flock(stream.fileno(), fcntl.LOCK_UN)\n        finally:\n            stream.close()\n\n    def create(self, session_id: str | None = None) -> SessionState:\n        accepted = _session_id(session_id or self._id_factory())\n        path = self.path_for(accepted)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        record = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "session",\n            "session_id": accepted,\n        }\n        encoded = json.dumps(\n            record, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n        ) + "\\n"\n        try:\n            with path.open("x", encoding="utf-8") as stream:\n                stream.write(encoded)\n                stream.flush()\n                os.fsync(stream.fileno())\n        except FileExistsError:\n            raise ValueError(f"Session already exists: {accepted}") from None\n        return SessionState(accepted)\n\n    def read(self, session_id: str) -> SessionState:\n        accepted = _session_id(session_id)\n        path = self.path_for(accepted)\n        try:\n            content = path.read_bytes()\n        except FileNotFoundError:\n            raise KeyError(f"unknown Session: {accepted}") from None\n        raw_lines = content.splitlines(keepends=True)\n        warning = None\n        if raw_lines and not raw_lines[-1].endswith(b"\\n"):\n            warning = RecoveryWarning(\n                RecoveryCode.INCOMPLETE_FINAL_RECORD,\n                "ignored an incomplete final Session record; restart the operation",\n                len(raw_lines),\n            )\n            raw_lines = raw_lines[:-1]\n        try:\n            lines = [line.decode("utf-8").rstrip("\\r\\n") for line in raw_lines]\n        except UnicodeDecodeError as error:\n            raise ValueError("Session JSONL must be UTF-8 text") from error\n        if not lines:\n            raise ValueError("Session file has no committed header")\n        entries: list[SessionEntry] = []\n        for line_number, line in enumerate(lines, 1):\n            try:\n                decoded = json.loads(line)\n            except json.JSONDecodeError as error:\n                raise ValueError(\n                    f"invalid Session JSONL record at line {line_number}"\n                ) from error\n            record = _validate_record_version(decoded, line_number)\n            if line_number == 1:\n                if record.get("record") != "session" or record.get("session_id") != accepted:\n                    raise ValueError("invalid Session header")\n                continue\n            if record.get("session_id") != accepted:\n                raise ValueError(f"invalid Session record at line {line_number}")\n            if record.get("record") == "custom":\n                raw_custom = record.get("custom")\n                if not isinstance(raw_custom, dict):\n                    raise ValueError("invalid Custom Entry record")\n                entries.append(\n                    SessionEntry(\n                        str(record["entry_id"]),\n                        (\n                            None\n                            if record.get("parent_id") is None\n                            else str(record["parent_id"])\n                        ),\n                        custom=_decode_custom_entry(raw_custom),\n                    )\n                )\n                continue\n            if record.get("record") == "compaction":\n                raw_checkpoint = record.get("checkpoint")\n                if not isinstance(raw_checkpoint, dict):\n                    raise ValueError("invalid Compaction checkpoint record")\n                entries.append(\n                    SessionEntry(\n                        str(record["entry_id"]),\n                        (\n                            None\n                            if record.get("parent_id") is None\n                            else str(record["parent_id"])\n                        ),\n                        compaction=_decode_compaction(raw_checkpoint),\n                    )\n                )\n                continue\n            if record.get("record") != "settlement":\n                raise ValueError(f"invalid Session record at line {line_number}")\n            raw_messages = record.get("messages")\n            if not isinstance(raw_messages, list) or not raw_messages:\n                raise ValueError("a settled Session entry requires messages")\n            entries.append(\n                SessionEntry(\n                    str(record["entry_id"]),\n                    None if record.get("parent_id") is None else str(record["parent_id"]),\n                    tuple(_decode_message(message) for message in raw_messages),\n                )\n            )\n        state = SessionState(accepted, tuple(entries), warning)\n        state.history()\n        return state\n\n    def writer(self, session_id: str) -> JSONLSessionWriter:\n        self.read(session_id)\n        return JSONLSessionWriter(self, session_id)\n\n\ndef migrate_session_file(\n    source: str | Path,\n    destination: str | Path,\n) -> MigrationResult:\n    """Write and validate a current Session file without changing its source."""\n\n    source_path = Path(source).resolve()\n    destination_path = Path(destination).resolve()\n    if source_path.parent.name != "sessions":\n        raise ValueError("source must be a file from a sessions directory")\n    session_id = _session_id(source_path.stem)\n    if destination_path.name != f"{session_id}.jsonl":\n        raise ValueError("migration destination must retain the Session filename")\n    if destination_path.exists():\n        raise FileExistsError(f"migration destination exists: {destination_path}")\n    state = JSONLSessionStore(source_path.parent.parent).read(session_id)\n    records: list[dict[str, object]] = [\n        {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "record": "session",\n            "session_id": session_id,\n        }\n    ]\n    for entry in state.entries:\n        common: dict[str, object] = {\n            "schema": "agent_harness.session",\n            "schema_version": _version_record(),\n            "session_id": session_id,\n            "entry_id": entry.entry_id,\n            "parent_id": entry.parent_id,\n        }\n        if entry.custom is not None:\n            records.append(\n                {\n                    **common,\n                    "record": "custom",\n                    "custom": _encode_custom_entry(entry.custom),\n                }\n            )\n        elif entry.compaction is not None:\n            records.append(\n                {\n                    **common,\n                    "record": "compaction",\n                    "checkpoint": _encode_compaction(entry.compaction),\n                }\n            )\n        else:\n            records.append(\n                {\n                    **common,\n                    "record": "settlement",\n                    "messages": [\n                        _encode_message(message) for message in entry.messages\n                    ],\n                }\n            )\n    encoded = "".join(\n        json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n        + "\\n"\n        for record in records\n    )\n    destination_path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(\n        prefix=".session-migration-", dir=destination_path.parent.parent\n    ) as temporary:\n        staging_root = Path(temporary)\n        staging = staging_root / "sessions" / f"{session_id}.jsonl"\n        staging.parent.mkdir(parents=True)\n        with staging.open("x", encoding="utf-8") as stream:\n            stream.write(encoded)\n            stream.flush()\n            os.fsync(stream.fileno())\n        validated = JSONLSessionStore(staging_root).read(session_id)\n        if (\n            validated.history() != state.history()\n            or validated.compactions() != state.compactions()\n            or validated.custom_entries() != state.custom_entries()\n        ):\n            raise ValueError("migrated Session failed history validation")\n        staging.replace(destination_path)\n    return MigrationResult(\n        source_path,\n        destination_path,\n        session_id,\n        len(state.entries),\n    )\n'


In [ ]:
SESSION_SOURCE = '"""Application-facing control for one in-memory agent conversation."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections import deque\nfrom dataclasses import dataclass\nfrom collections.abc import AsyncIterator, Mapping, Sequence\nfrom enum import Enum\nfrom typing import cast\n\nfrom .compaction import (\n    CharacterTokenEstimator,\n    CompactionCheckpoint,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarning,\n    CompactionWarningCode,\n    StructuredSummary,\n    TokenEstimator,\n)\nfrom .extensions import (\n    CompactionHookRequest,\n    CommandExecutionError,\n    Extension,\n    ExtensionHost,\n    HookPoint,\n    ReloadResult,\n)\nfrom .model import AgentMessage, Role, StopReason\nfrom .persistence import SessionBusyError, SessionStore, SessionWriter\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    RunSnapshot,\n    RuntimeEvent,\n)\n\n\n_SESSION_EVENTS_DONE = object()\n\n\nclass InputKind(str, Enum):\n    STEERING = "steering"\n    FOLLOW_UP = "follow_up"\n\n\n@dataclass(frozen=True, slots=True)\nclass PendingInput:\n    kind: InputKind\n    message: AgentMessage\n\n\n@dataclass(frozen=True, slots=True)\nclass SessionRunResult:\n    outcome: AssistantOutcome\n    outcomes: tuple[AssistantOutcome, ...] = ()\n    pending_inputs: tuple[PendingInput, ...] = ()\n\n\n@dataclass(frozen=True, slots=True)\nclass CompactionResult:\n    checkpoint: CompactionCheckpoint\n\n\nclass SessionRunHandle:\n    def __init__(\n        self,\n        task: asyncio.Task[SessionRunResult],\n        session: "AgentSession",\n        events: asyncio.Queue[RuntimeEvent | object],\n        snapshot: RunSnapshot,\n    ) -> None:\n        self._task = task\n        self._session = session\n        self._events = events\n        self._snapshot = snapshot\n\n    @property\n    def run_id(self) -> str:\n        return self._snapshot.run_id\n\n    @property\n    def snapshot(self) -> RunSnapshot:\n        return self._snapshot\n\n    async def result(self) -> SessionRunResult:\n        return await self._task\n\n    def cancel(self) -> None:\n        if not self._task.done():\n            self._session.cancel()\n\n    async def events(self) -> AsyncIterator[RuntimeEvent]:\n        while True:\n            event = await self._events.get()\n            if event is _SESSION_EVENTS_DONE:\n                break\n            yield cast(RuntimeEvent, event)\n\n\nclass _SteeringQueue:\n    def __init__(self) -> None:\n        self._messages: deque[PendingInput] = deque()\n\n    def append(self, message: AgentMessage) -> None:\n        self._messages.append(PendingInput(InputKind.STEERING, message))\n\n    def pending(self) -> bool:\n        return bool(self._messages)\n\n    def take(self) -> Sequence[AgentMessage]:\n        return (self._messages.popleft().message,)\n\n    def drain(self) -> tuple[PendingInput, ...]:\n        drained = tuple(self._messages)\n        self._messages.clear()\n        return drained\n\n\nclass AgentSession:\n    """Coordinate one active Run with optional durable Session state."""\n\n    def __init__(\n        self,\n        runtime: AgentRuntime,\n        *,\n        store: SessionStore | None = None,\n        session_id: str | None = None,\n        parent_entry_id: str | None = None,\n        compaction_policy: CompactionPolicy | None = None,\n        compaction_strategy: CompactionStrategy | None = None,\n        token_estimator: TokenEstimator | None = None,\n        extensions: Sequence[Extension] = (),\n        prompt_hashes: Mapping[str, str] | None = None,\n        resource_hashes: Mapping[str, str] | None = None,\n    ) -> None:\n        if not isinstance(runtime, AgentRuntime):\n            raise TypeError("runtime must be an AgentRuntime")\n        if (store is None) != (session_id is None):\n            raise ValueError("durable Sessions require both store and session_id")\n        if compaction_policy is not None and not isinstance(\n            compaction_policy, CompactionPolicy\n        ):\n            raise TypeError("compaction_policy must be a CompactionPolicy")\n        if compaction_strategy is not None and not callable(\n            getattr(compaction_strategy, "plan", None)\n        ):\n            raise TypeError("compaction_strategy must provide plan()")\n        if token_estimator is not None and not callable(\n            getattr(token_estimator, "estimate", None)\n        ):\n            raise TypeError("token_estimator must provide estimate()")\n        self._runtime = runtime\n        self._base_tools = runtime.tools\n        self._extension_host = ExtensionHost(self._base_tools, extensions)\n        runtime.configure_tools(self._extension_host.tools)\n        runtime.configure_subscribers(self._extension_host.subscribers)\n        runtime.configure_hooks(self._extension_host.hooks)\n        self._store = store\n        self._session_id = session_id\n        self._parent_entry_id = parent_entry_id\n        self._compaction_policy = compaction_policy or CompactionPolicy()\n        self._compaction_strategy = compaction_strategy or CompactionStrategy()\n        self._token_estimator = token_estimator or CharacterTokenEstimator()\n        self._prompt_hashes = dict(prompt_hashes or {})\n        self._resource_hashes = dict(resource_hashes or {})\n        if any(\n            not isinstance(name, str)\n            or not name\n            or not isinstance(digest, str)\n            or not digest\n            for name, digest in (\n                *self._prompt_hashes.items(),\n                *self._resource_hashes.items(),\n            )\n        ):\n            raise ValueError("snapshot hashes require non-empty string names and values")\n        self._flush_pending_custom_entries(self._extension_host)\n        self._warnings: list[CompactionWarning] = []\n        self._busy = False\n        self._active: AgentRunHandle | None = None\n        self._compaction_task: asyncio.Task[object] | None = None\n        self._cancel_requested = False\n        self._steering = _SteeringQueue()\n        self._follow_ups: deque[PendingInput] = deque()\n\n    @property\n    def busy(self) -> bool:\n        return self._busy\n\n    @property\n    def session_id(self) -> str | None:\n        return self._session_id\n\n    @property\n    def warnings(self) -> tuple[CompactionWarning, ...]:\n        return tuple(self._warnings)\n\n    @property\n    def extension_replacements(self):\n        return self._extension_host.replacements\n\n    def run_annotations(self, run_id: str | None = None):\n        return self._extension_host.run_annotations(run_id)\n\n    def _flush_pending_custom_entries(self, host: ExtensionHost) -> None:\n        if self._store is None:\n            return\n        assert self._session_id is not None\n        with self._store.writer(self._session_id) as writer:\n            def persist_custom(custom) -> None:\n                entry = writer.append_custom(\n                    custom,\n                    parent_id=self._parent_entry_id,\n                )\n                self._parent_entry_id = entry.entry_id\n\n            host.activate_custom_entry_sink(persist_custom)\n            host.deactivate_custom_entry_sink()\n\n    def start(self, prompt: str | AgentMessage) -> SessionRunHandle:\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Session already has an active Run",\n            )\n        message = (\n            AgentMessage.text(Role.USER, prompt) if isinstance(prompt, str) else prompt\n        )\n        if not isinstance(message, AgentMessage) or message.role is not Role.USER:\n            raise TypeError("prompt must be text or a user AgentMessage")\n        writer: SessionWriter | None = None\n        if self._store is not None:\n            assert self._session_id is not None\n            writer = self._store.writer(self._session_id)\n            writer = writer.__enter__()\n        self._busy = True\n        self._cancel_requested = False\n        events: asyncio.Queue[RuntimeEvent | object] = asyncio.Queue()\n        try:\n            snapshot = self._runtime.capture_snapshot(\n                extension_identities=self._extension_host.identities,\n                compaction_policy=self._compaction_policy,\n                prompt_hashes=self._prompt_hashes,\n                resource_hashes=self._resource_hashes,\n            )\n            task = asyncio.get_running_loop().create_task(\n                self._drive(message, events, writer, snapshot)\n            )\n        except BaseException:\n            self._busy = False\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            raise\n        return SessionRunHandle(task, self, events, snapshot)\n\n    async def run(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.start(prompt).result()\n\n    async def prompt(self, prompt: str | AgentMessage) -> SessionRunResult:\n        return await self.run(prompt)\n\n    async def execute_command(self, name: str, arguments: str = "") -> object:\n        """Execute one registered chat command without starting an Agent Run."""\n\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "chat commands require an idle Session",\n            )\n        registration = self._extension_host.command(name)\n        try:\n            return await registration.handler(arguments)\n        except Exception as error:\n            raise CommandExecutionError(name, type(error).__name__) from error\n\n    async def reload_extensions(\n        self,\n        extensions: Sequence[Extension],\n    ) -> ReloadResult:\n        """Replace explicit Extensions only while the Session is idle."""\n\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Extension reload requires an idle Session",\n            )\n        self._busy = True\n        old_host = self._extension_host\n        try:\n            warnings = await old_host.teardown()\n            replacement = ExtensionHost(self._base_tools, extensions)\n            old_names = {extension.name for extension in old_host.extensions}\n            new_names = {extension.name for extension in replacement.extensions}\n            self._extension_host = replacement\n            self._runtime.configure_tools(replacement.tools)\n            self._runtime.configure_subscribers(replacement.subscribers)\n            self._runtime.configure_hooks(replacement.hooks)\n            self._flush_pending_custom_entries(replacement)\n            return ReloadResult(\n                tuple(sorted(old_names & new_names)),\n                warnings,\n                replacement.replacements,\n            )\n        finally:\n            self._busy = False\n\n    async def compact(self, focus: str | None = None) -> CompactionResult:\n        """Create and install a context checkpoint at a Settled Boundary."""\n\n        if focus is not None and not focus.strip():\n            raise ValueError("Compaction focus cannot be empty")\n        if self._busy:\n            raise SessionBusyError(\n                self._session_id or "ephemeral",\n                "Compaction requires an idle Session at a Settled Boundary",\n            )\n        writer: SessionWriter | None = None\n        if self._store is not None:\n            assert self._session_id is not None\n            writer = self._store.writer(self._session_id).__enter__()\n        self._busy = True\n        try:\n            snapshot = self._runtime.capture_snapshot(\n                extension_identities=self._extension_host.identities,\n                compaction_policy=self._compaction_policy,\n                prompt_hashes=self._prompt_hashes,\n                resource_hashes=self._resource_hashes,\n            )\n            return await self._compact(\n                CompactionTrigger.MANUAL,\n                focus=focus,\n                writer=writer,\n                snapshot=snapshot,\n            )\n        finally:\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            self._busy = False\n\n    async def _compact(\n        self,\n        trigger: CompactionTrigger,\n        *,\n        focus: str | None,\n        writer: SessionWriter | None,\n        snapshot: RunSnapshot | None = None,\n    ) -> CompactionResult:\n        current = asyncio.current_task()\n        previous = self._compaction_task\n        self._compaction_task = cast(asyncio.Task[object] | None, current)\n        try:\n            hook_request = await self._runtime.apply_hooks(\n                HookPoint.BEFORE_COMPACTION,\n                CompactionHookRequest(trigger, focus),\n                snapshot=snapshot,\n            )\n            if not isinstance(hook_request, CompactionHookRequest):\n                raise TypeError(\n                    "before_compaction Hook must return CompactionHookRequest or None"\n                )\n            if hook_request.trigger is not trigger:\n                raise ValueError("before_compaction Hook cannot change its trigger")\n            focus = hook_request.focus\n            configured_policy = (\n                snapshot.compaction_policy\n                if snapshot is not None\n                and isinstance(snapshot.compaction_policy, CompactionPolicy)\n                else self._compaction_policy\n            )\n            policy = configured_policy.resolve(\n                self._runtime.model if snapshot is None else snapshot.model\n            )\n            plan = self._compaction_strategy.plan(\n                self._runtime.effective_history,\n                keep_recent_tokens=policy.keep_recent_tokens,\n                estimator=self._token_estimator,\n            )\n            generated = await self._runtime.generate_summary(\n                plan.source,\n                focus=focus,\n                snapshot=snapshot,\n            )\n            checkpoint = CompactionCheckpoint(\n                trigger=trigger,\n                summary=StructuredSummary(generated.text, focus=focus),\n                tokens_before=plan.tokens_before,\n                summary_usage=generated.usage,\n                retained_tail=plan.retained_tail,\n            )\n            if writer is not None:\n                entry = writer.append_compaction(\n                    checkpoint,\n                    parent_id=self._parent_entry_id,\n                )\n                self._parent_entry_id = entry.entry_id\n            self._runtime.install_compaction(checkpoint)\n            return CompactionResult(checkpoint)\n        finally:\n            self._compaction_task = previous\n\n    async def _maybe_compact_after_settlement(\n        self,\n        writer: SessionWriter | None,\n        snapshot: RunSnapshot | None = None,\n    ) -> CompactionResult | None:\n        configured_policy = (\n            snapshot.compaction_policy\n            if snapshot is not None\n            and isinstance(snapshot.compaction_policy, CompactionPolicy)\n            else self._compaction_policy\n        )\n        resolved = configured_policy.resolve(\n            self._runtime.model if snapshot is None else snapshot.model\n        )\n        threshold = resolved.threshold_tokens\n        if threshold is None:\n            if (\n                resolved.context_window is None\n                and not any(\n                    warning.code is CompactionWarningCode.CONTEXT_WINDOW_UNKNOWN\n                    for warning in self._warnings\n                )\n            ):\n                self._warnings.append(\n                    CompactionWarning(\n                        CompactionWarningCode.CONTEXT_WINDOW_UNKNOWN,\n                        "threshold Compaction disabled because ModelSpec has no "\n                        "context_window; normalized overflow recovery remains enabled",\n                    )\n                )\n            return None\n        current_tokens = self._token_estimator.estimate(\n            self._runtime.effective_history\n        )\n        if current_tokens <= threshold:\n            return None\n        try:\n            return await self._compact(\n                CompactionTrigger.THRESHOLD,\n                focus=None,\n                writer=writer,\n                snapshot=snapshot,\n            )\n        except Exception as error:\n            self._warnings.append(\n                CompactionWarning(\n                    CompactionWarningCode.THRESHOLD_FAILED,\n                    "threshold Compaction failed; Session history unchanged: "\n                    f"{type(error).__name__}",\n                )\n            )\n            return None\n\n    def steer(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Steering requires an active Run")\n        self._steering.append(self._user_message(message))\n\n    def follow_up(self, message: str | AgentMessage) -> None:\n        if not self._busy:\n            raise RuntimeError("Follow-up requires an active Run")\n        self._follow_ups.append(\n            PendingInput(InputKind.FOLLOW_UP, self._user_message(message))\n        )\n\n    def cancel(self) -> None:\n        self._cancel_requested = True\n        if self._compaction_task is not None and not self._compaction_task.done():\n            self._compaction_task.cancel()\n        if self._active is not None:\n            self._active.cancel()\n\n    async def _drive(\n        self,\n        message: AgentMessage,\n        events: asyncio.Queue[RuntimeEvent | object],\n        writer: SessionWriter | None,\n        snapshot: RunSnapshot,\n    ) -> SessionRunResult:\n        outcomes: list[AssistantOutcome] = []\n\n        if writer is not None:\n            def settle(messages) -> None:\n                entry = writer.append(messages, parent_id=self._parent_entry_id)\n                self._parent_entry_id = entry.entry_id\n\n            self._runtime.set_settlement_sink(settle)\n\n            def persist_custom(custom) -> None:\n                entry = writer.append_custom(\n                    custom,\n                    parent_id=self._parent_entry_id,\n                )\n                self._parent_entry_id = entry.entry_id\n\n            self._extension_host.activate_custom_entry_sink(persist_custom)\n\n        async def recover_context_overflow() -> bool:\n            await self._compact(\n                CompactionTrigger.OVERFLOW,\n                focus=None,\n                writer=writer,\n                snapshot=snapshot,\n            )\n            return True\n\n        self._runtime.set_context_overflow_recovery(recover_context_overflow)\n        self._runtime.set_settled_turn_handler(\n            lambda: self._maybe_compact_after_settlement(writer, snapshot)\n        )\n\n        async def forward(handle: AgentRunHandle) -> None:\n            async for event in handle.events():\n                await events.put(event)\n\n        try:\n            await self._extension_host.startup()\n            next_message = message\n            while True:\n                self._active = self._runtime.start(\n                    [next_message],\n                    turn_input=self._steering,\n                    snapshot=snapshot,\n                )\n                if self._cancel_requested:\n                    self._active.cancel()\n                forwarding = asyncio.create_task(forward(self._active))\n                outcome = await self._active.result()\n                await forwarding\n                outcomes.append(outcome)\n                if outcome.stop_reason in {StopReason.ABORTED, StopReason.ERROR}:\n                    break\n                if not self._follow_ups:\n                    break\n                next_message = self._follow_ups.popleft().message\n            pending = self._steering.drain() + tuple(self._follow_ups)\n            self._follow_ups.clear()\n            return SessionRunResult(outcome, tuple(outcomes), pending)\n        finally:\n            self._runtime.set_settlement_sink(None)\n            self._runtime.set_context_overflow_recovery(None)\n            self._runtime.set_settled_turn_handler(None)\n            self._extension_host.deactivate_custom_entry_sink()\n            if writer is not None:\n                writer.__exit__(None, None, None)\n            self._active = None\n            self._cancel_requested = False\n            self._busy = False\n            await events.put(_SESSION_EVENTS_DONE)\n\n    @staticmethod\n    def _user_message(message: str | AgentMessage) -> AgentMessage:\n        accepted = (\n            AgentMessage.text(Role.USER, message)\n            if isinstance(message, str)\n            else message\n        )\n        if not isinstance(accepted, AgentMessage) or accepted.role is not Role.USER:\n            raise TypeError("Session input must be text or a user AgentMessage")\n        return accepted\n\n\ndef create_agent_session(\n    runtime: AgentRuntime,\n    *,\n    store: SessionStore | None = None,\n    session_id: str | None = None,\n    fork_from: str | None = None,\n    no_save: bool = False,\n    compaction_policy: CompactionPolicy | None = None,\n    compaction_strategy: CompactionStrategy | None = None,\n    token_estimator: TokenEstimator | None = None,\n    extensions: Sequence[Extension] = (),\n    prompt_hashes: Mapping[str, str] | None = None,\n    resource_hashes: Mapping[str, str] | None = None,\n) -> AgentSession:\n    """Create a new durable Session, explicitly continue/fork one, or opt out."""\n\n    if no_save:\n        if store is not None or session_id is not None or fork_from is not None:\n            raise ValueError("no_save cannot be combined with persistence or continuation")\n        return AgentSession(\n            runtime,\n            compaction_policy=compaction_policy,\n            compaction_strategy=compaction_strategy,\n            token_estimator=token_estimator,\n            extensions=extensions,\n            prompt_hashes=prompt_hashes,\n            resource_hashes=resource_hashes,\n        )\n    if store is None:\n        if session_id is not None or fork_from is not None:\n            raise ValueError("continuation requires a SessionStore")\n        return AgentSession(\n            runtime,\n            compaction_policy=compaction_policy,\n            compaction_strategy=compaction_strategy,\n            token_estimator=token_estimator,\n            extensions=extensions,\n            prompt_hashes=prompt_hashes,\n            resource_hashes=resource_hashes,\n        )\n    if runtime.history:\n        raise ValueError("a durable AgentSession requires a fresh AgentRuntime")\n    if session_id is None:\n        if fork_from is not None:\n            raise ValueError("fork_from requires an existing session_id")\n        state = store.create()\n        leaf = None\n    else:\n        state = store.read(session_id)\n        leaf = fork_from if fork_from is not None else state.active_leaf_id\n        runtime.restore_history(\n            state.history(leaf),\n            effective_history=state.effective_history(leaf),\n        )\n    return AgentSession(\n        runtime,\n        store=store,\n        session_id=state.session_id,\n        parent_entry_id=leaf,\n        compaction_policy=compaction_policy,\n        compaction_strategy=compaction_strategy,\n        token_estimator=token_estimator,\n        extensions=extensions,\n        prompt_hashes=prompt_hashes,\n        resource_hashes=resource_hashes,\n    )\n'


In [ ]:
INIT_SOURCE = 'from .compaction import (\n    CharacterTokenEstimator,\n    CompactionCheckpoint,\n    CompactionPlan,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarning,\n    CompactionWarningCode,\n    ResolvedCompactionPolicy,\n    StructuredSummary,\n    TokenEstimator,\n)\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelMessage,\n    ModelOperation,\n    ModelProtocolError,\n    ModelRequest,\n    ModelResult,\n    ModelSpec,\n    ModelToolResultMessage,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n    ScriptedModelAdapter,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    UnsupportedContentError,\n    Usage,\n    UsageUpdate,\n    complete,\n    to_model_messages,\n)\nfrom .extensions import (\n    Extension,\n    ExtensionAPI,\n    ExtensionHost,\n    ExtensionInitializationError,\n    SubscriberRegistration,\n    HookContext,\n    HookExecutionError,\n    HookPoint,\n    HookRegistration,\n    CompactionHookRequest,\n    CommandExecutionError,\n    CommandRegistration,\n    CustomEntry,\n    RunAnnotation,\n    LifecycleWarning,\n    ReloadResult,\n    ReplacementRecord,\n)\nfrom .persistence import (\n    ConversationMessage,\n    JSONLSessionStore,\n    JSONLSessionWriter,\n    MemorySessionStore,\n    MemorySessionWriter,\n    MigrationResult,\n    RecoveryCode,\n    RecoveryWarning,\n    SESSION_SCHEMA_VERSION,\n    SchemaVersion,\n    SessionBusyError,\n    SessionEntry,\n    SessionStore,\n    SessionState,\n    SessionWriter,\n    UnsupportedSchemaVersionError,\n    migrate_session_file,\n)\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    EventType,\n    RetryPolicy,\n    RunSnapshot,\n    RunGuard,\n    RuntimeEvent,\n    Sleeper,\n    ContextOverflowRecovery,\n    SettledTurnHandler,\n    SummaryGeneration,\n    TerminalStatus,\n    TurnInput,\n)\nfrom .session import (\n    AgentSession,\n    CompactionResult,\n    InputKind,\n    PendingInput,\n    SessionRunHandle,\n    SessionRunResult,\n    create_agent_session,\n)\nfrom .tools import (\n    AsyncioProcessOperations,\n    CompleteOutputKind,\n    CompleteOutputReference,\n    LocalToolExecutor,\n    PreparedToolCall,\n    ProcessResult,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\n\n__all__ = [name for name in globals() if not name.startswith("_")]\n'


In [ ]:
import asyncio
import importlib
import shutil
from tempfile import TemporaryDirectory

temporary_package = TemporaryDirectory(prefix='chapter-07-minimal-')
package = Path(temporary_package.name) / 'agent_harness'
shutil.copytree(CHAPTER_6 / 'src' / 'agent_harness', package)
for name, text in {
    'extensions.py': EXTENSIONS_SOURCE, 'model.py': MODEL_SOURCE,
    'tools.py': TOOLS_SOURCE, 'runtime.py': RUNTIME_SOURCE,
    'persistence.py': PERSISTENCE_SOURCE, 'session.py': SESSION_SOURCE,
    '__init__.py': INIT_SOURCE,
}.items():
    (package / name).write_text(text, encoding='utf-8')
sys.path.insert(0, temporary_package.name)
chapter7 = importlib.import_module('agent_harness')

minimal_observations = []
def configure_minimal(api):
    async def observe(event):
        if event.type in {chapter7.EventType.MESSAGE_END, chapter7.EventType.AGENT_END}:
            minimal_observations.append(event.type.value)
    api.subscribe(observe)

minimal_session = chapter7.AgentSession(
    chapter7.AgentRuntime(
        chapter7.ScriptedModelAdapter([
            chapter7.TextDelta('extended'),
            chapter7.ModelEnd(chapter7.StopReason.COMPLETE),
        ]),
        chapter7.ModelSpec('scripted/ch07-minimal'),
    ),
    extensions=[chapter7.Extension('observer', '1', configure_minimal)],
)
minimal_result = await minimal_session.run('observe this Run')
assert minimal_result.outcome.message.content[0].text == 'extended'
assert minimal_observations == ['message_end', 'agent_end']


## Staged Construction

Registration is deterministic and collision-safe. Custom Entries use the Extension's namespace and a positive schema version; they remain outside model history. Hooks receive the frozen snapshot, so annotations correlate with the accepted Run rather than mutable Session configuration.

In [ ]:
annotation_session_ref = {}
def configure_state(api):
    api.append_custom_entry('state.cursor', 1, {'position': 2})
    async def annotate(context):
        api.add_run_annotation(
            context.snapshot.run_id, 'state.review', 1, {'accepted': True}
        )
    api.register_hook(chapter7.HookPoint.BEFORE_RUN, annotate)

state_session = chapter7.AgentSession(
    chapter7.AgentRuntime(
        chapter7.ScriptedModelAdapter([
            chapter7.TextDelta('stateful'),
            chapter7.ModelEnd(chapter7.StopReason.COMPLETE),
        ]),
        chapter7.ModelSpec('scripted/ch07-state'),
        generation_settings={'temperature': 0},
    ),
    extensions=[chapter7.Extension('state', '1', configure_state)],
    prompt_hashes={'system': 'sha256:prompt'},
    resource_hashes={'skill:state': 'sha256:skill'},
)
state_handle = state_session.start('capture state')
await state_handle.result()
assert state_session.run_annotations(state_handle.run_id)[0].namespace == 'state.review'
assert state_handle.snapshot.resource_hashes == (('skill:state', 'sha256:skill'),)


## Observable Trace

Every Event carries the Run id and snapshot fingerprint. The iterator can be consumed after completion, demonstrating that consumer work is not a Runtime barrier.

In [ ]:
trace_adapter = chapter7.ScriptedModelAdapter([
    chapter7.TextDelta('traceable'),
    chapter7.ModelEnd(chapter7.StopReason.COMPLETE),
])
trace_session = chapter7.AgentSession(
    chapter7.AgentRuntime(trace_adapter, chapter7.ModelSpec('scripted/ch07-trace'))
)
trace_handle = trace_session.start('trace')
await trace_handle.result()
trace_events = [event async for event in trace_handle.events()]
assert [event.sequence for event in trace_events] == list(range(1, len(trace_events) + 1))
assert all(event.run_id == trace_handle.run_id for event in trace_events)
[(event.sequence, event.type.value) for event in trace_events]


## Failure Boundaries and Trade-offs

Subscriber failures are isolated and surfaced as `subscriber_failed`; other subscribers and the semantic Run continue. Hook failures instead fail safe: Run/model Hooks terminate, Tool Hooks block or withhold, and Compaction Hooks cancel. Trusted Extensions retain host-process authority; core provides no sandbox or implicit loader.

In [ ]:
def configure_broken_observer(api):
    async def broken(event):
        if event.type is chapter7.EventType.AGENT_START:
            raise RuntimeError('observer failure')
    api.subscribe(broken)

failure_session = chapter7.AgentSession(
    chapter7.AgentRuntime(
        chapter7.ScriptedModelAdapter([
            chapter7.TextDelta('still succeeds'),
            chapter7.ModelEnd(chapter7.StopReason.COMPLETE),
        ]),
        chapter7.ModelSpec('scripted/ch07-failure'),
    ),
    extensions=[chapter7.Extension('broken-observer', '1', configure_broken_observer)],
)
failure_handle = failure_session.start('isolate')
failure_result = await failure_handle.result()
failure_events = [event async for event in failure_handle.events()]
assert failure_result.outcome.stop_reason is chapter7.StopReason.COMPLETE
assert any(event.type is chapter7.EventType.SUBSCRIBER_FAILED for event in failure_events)


## Checkpoint Export and Verification

Chapter metadata names Chapter 6 as the base Checkpoint. Export carries forward unchanged modules and all prior regressions, replaces evolved files, adds Chapter 7 tests, writes a deterministic manifest, then compiles, installs offline, imports, and runs the cumulative suite before publishing.

In [ ]:
EXTENSION_TEST_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nfrom dataclasses import replace\n\nimport pytest\n\nfrom agent_harness import (\n    AgentRuntime,\n    AgentSession,\n    Extension,\n    ModelEnd,\n    ModelSpec,\n    ScriptedModelAdapter,\n    StopReason,\n    TextDelta,\n    Tool,\n    ToolCallDelta,\n    ToolResult,\n    ToolErrorCode,\n    EventType,\n    MemorySessionStore,\n    AgentMessage,\n    HookPoint,\n    ModelRequest,\n    Role,\n    HookExecutionError,\n    SessionBusyError,\n    ExtensionInitializationError,\n    JSONLSessionStore,\n    CompactionPolicy,\n    RetryPolicy,\n    RunGuard,\n    CommandExecutionError,\n    CompactionWarningCode,\n    ModelAdapterError,\n    ModelError,\n    ModelErrorCode,\n)\n\n\ndef test_explicit_extension_registers_a_tool_for_the_session() -> None:\n    async def scenario() -> None:\n        calls: list[dict[str, object]] = []\n\n        async def inspect(arguments: dict[str, object]) -> ToolResult:\n            calls.append(arguments)\n            return ToolResult("extension result")\n\n        def configure(api) -> None:\n            api.register_tool(\n                Tool(\n                    "inspect",\n                    "Inspect one value",\n                    {\n                        "type": "object",\n                        "properties": {"value": {"type": "string"}},\n                        "required": ["value"],\n                        "additionalProperties": False,\n                    },\n                    inspect,\n                )\n            )\n\n        adapter = ScriptedModelAdapter(\n            [\n                [\n                    ToolCallDelta(0, "call-1", "inspect", \'{"value":"ready"}\'),\n                    ModelEnd(StopReason.TOOL_USE),\n                ],\n                [TextDelta("done"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/extensions")),\n            extensions=[Extension("example", "1.0.0", configure)],\n        )\n\n        result = await session.run("use the extension")\n\n        assert result.outcome.message.content[0].text == "done"\n        assert calls == [{"value": "ready"}]\n\n    asyncio.run(scenario())\n\n\ndef test_high_level_event_subscribers_are_ordered_lifecycle_barriers() -> None:\n    async def scenario() -> None:\n        observed: list[str] = []\n        store = MemorySessionStore(id_factory=iter(("session-1", "entry-1", "entry-2")).__next__)\n        state = store.create()\n        session: AgentSession\n\n        def configure_first(api) -> None:\n            async def observe(event) -> None:\n                if event.type is EventType.MESSAGE_END:\n                    assert len(store.read(state.session_id).history()) == 2\n                    observed.append("first:message_end")\n                if event.type is EventType.AGENT_END:\n                    assert session.busy is True\n                    observed.append("first:agent_end")\n\n            api.subscribe(observe)\n\n        def configure_second(api) -> None:\n            async def observe(event) -> None:\n                if event.type in {EventType.MESSAGE_END, EventType.AGENT_END}:\n                    await asyncio.sleep(0)\n                    observed.append(f"second:{event.type.value}")\n\n            api.subscribe(observe)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("settled"), ModelEnd(StopReason.COMPLETE)]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/event-barriers")),\n            store=store,\n            session_id=state.session_id,\n            extensions=[\n                Extension("first", "1", configure_first),\n                Extension("second", "1", configure_second),\n            ],\n        )\n\n        await session.run("persist before observing")\n\n        assert observed == [\n            "first:message_end",\n            "second:message_end",\n            "first:agent_end",\n            "second:agent_end",\n        ]\n        assert session.busy is False\n\n    asyncio.run(scenario())\n\n\ndef test_low_level_event_iteration_is_ordered_but_not_a_runtime_barrier() -> None:\n    async def scenario() -> None:\n        session = AgentSession(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("complete"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/observational-stream"),\n            )\n        )\n\n        handle = session.start("do not wait for my consumer")\n        result = await asyncio.wait_for(handle.result(), timeout=1)\n        events = [event async for event in handle.events()]\n\n        assert result.outcome.stop_reason is StopReason.COMPLETE\n        assert [event.sequence for event in events] == list(\n            range(1, len(events) + 1)\n        )\n        assert events[-1].type is EventType.AGENT_END\n\n    asyncio.run(scenario())\n\n\ndef test_before_model_request_hook_can_transform_the_pending_request() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def add_context(context):\n                request = context.value\n                assert isinstance(request, ModelRequest)\n                return replace(\n                    request,\n                    messages=(\n                        *request.messages,\n                        AgentMessage.text(Role.SYSTEM, "extension context").to_model(),\n                    ),\n                )\n\n            api.register_hook(HookPoint.BEFORE_MODEL_REQUEST, add_context)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("hooked"), ModelEnd(StopReason.COMPLETE)]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/before-model-hook")),\n            extensions=[Extension("request-context", "1", configure)],\n        )\n\n        await session.run("start")\n\n        assert adapter.received_requests[0].messages[-1].content[0].text == (\n            "extension context"\n        )\n\n    asyncio.run(scenario())\n\n\ndef test_before_run_hook_failure_terminates_without_model_work() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def reject(context):\n                raise RuntimeError("policy unavailable")\n\n            api.register_hook(HookPoint.BEFORE_RUN, reject)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("must not run"), ModelEnd(StopReason.COMPLETE)]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/before-run-failure")),\n            extensions=[Extension("guard", "1", configure)],\n        )\n\n        result = await session.run("blocked")\n\n        assert result.outcome.stop_reason is StopReason.ERROR\n        assert result.outcome.error is not None\n        assert result.outcome.error.code.value == "hook_failed"\n        assert "guard" in result.outcome.error.message\n        assert adapter.received_requests == ()\n        assert session.busy is False\n\n    asyncio.run(scenario())\n\n\ndef test_before_tool_call_hook_failure_blocks_tool_execution_safely() -> None:\n    async def scenario() -> None:\n        executed = False\n\n        async def dangerous(arguments: dict[str, object]) -> ToolResult:\n            nonlocal executed\n            executed = True\n            return ToolResult("unsafe")\n\n        def configure(api) -> None:\n            async def block(context):\n                raise PermissionError("approval unavailable")\n\n            api.register_hook(HookPoint.BEFORE_TOOL_CALL, block)\n\n        adapter = ScriptedModelAdapter(\n            [\n                [\n                    ToolCallDelta(0, "call-blocked", "dangerous", "{}"),\n                    ModelEnd(StopReason.TOOL_USE),\n                ],\n                [TextDelta("handled safely"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        session = AgentSession(\n            AgentRuntime(\n                adapter,\n                ModelSpec("scripted/before-tool-failure"),\n                tools=[\n                    Tool(\n                        "dangerous",\n                        "A guarded operation",\n                        {"type": "object"},\n                        dangerous,\n                    )\n                ],\n            ),\n            extensions=[Extension("approval", "1", configure)],\n        )\n\n        result = await session.run("try it")\n\n        assert executed is False\n        assert result.outcome.tool_results[0].error_code is ToolErrorCode.HOOK_FAILED\n        assert result.outcome.message.content[0].text == "handled safely"\n\n    asyncio.run(scenario())\n\n\ndef test_after_tool_call_hook_failure_withholds_the_unreviewed_result() -> None:\n    async def scenario() -> None:\n        async def reveal(arguments: dict[str, object]) -> ToolResult:\n            return ToolResult("unreviewed secret")\n\n        def configure(api) -> None:\n            async def fail_review(context):\n                raise RuntimeError("redactor failed")\n\n            api.register_hook(HookPoint.AFTER_TOOL_CALL, fail_review)\n\n        adapter = ScriptedModelAdapter(\n            [\n                [\n                    ToolCallDelta(0, "call-review", "reveal", "{}"),\n                    ModelEnd(StopReason.TOOL_USE),\n                ],\n                [TextDelta("safe continuation"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        session = AgentSession(\n            AgentRuntime(\n                adapter,\n                ModelSpec("scripted/after-tool-failure"),\n                tools=[\n                    Tool("reveal", "Return sensitive data", {"type": "object"}, reveal)\n                ],\n            ),\n            extensions=[Extension("redactor", "1", configure)],\n        )\n\n        result = await session.run("inspect")\n\n        tool_result = result.outcome.tool_results[0]\n        assert tool_result.error_code is ToolErrorCode.HOOK_FAILED\n        assert "unreviewed secret" not in adapter.received_requests[1].messages[-1].content\n\n    asyncio.run(scenario())\n\n\ndef test_before_compaction_hook_failure_cancels_manual_compaction() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def reject(context):\n                raise RuntimeError("summary policy unavailable")\n\n            api.register_hook(HookPoint.BEFORE_COMPACTION, reject)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("must not summarize"), ModelEnd(StopReason.COMPLETE)]\n        )\n        runtime = AgentRuntime(\n            adapter,\n            ModelSpec("scripted/compaction-hook"),\n            history=(\n                AgentMessage.text(Role.USER, "old question"),\n                AgentMessage.text(Role.ASSISTANT, "old answer"),\n            ),\n        )\n        session = AgentSession(\n            runtime,\n            extensions=[Extension("summary-policy", "1", configure)],\n        )\n\n        try:\n            await session.compact()\n        except HookExecutionError as error:\n            assert error.point is HookPoint.BEFORE_COMPACTION\n            assert error.extension_name == "summary-policy"\n        else:\n            raise AssertionError("manual Compaction must fail safe")\n\n        assert adapter.received_requests == ()\n        assert len(runtime.history) == 2\n        assert session.busy is False\n\n    asyncio.run(scenario())\n\n\ndef test_subscriber_failure_is_traced_and_does_not_change_the_run() -> None:\n    async def scenario() -> None:\n        reached_second: list[EventType] = []\n\n        def configure_failing(api) -> None:\n            async def fail(event) -> None:\n                if event.type is EventType.AGENT_START:\n                    raise RuntimeError("observer broke")\n\n            api.subscribe(fail)\n\n        def configure_healthy(api) -> None:\n            async def observe(event) -> None:\n                reached_second.append(event.type)\n\n            api.subscribe(observe)\n\n        session = AgentSession(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("still succeeds"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/subscriber-failure"),\n            ),\n            extensions=[\n                Extension("broken-observer", "1", configure_failing),\n                Extension("healthy-observer", "1", configure_healthy),\n            ],\n        )\n        handle = session.start("observe")\n        result = await handle.result()\n        events = [event async for event in handle.events()]\n\n        failures = [\n            event for event in events if event.type is EventType.SUBSCRIBER_FAILED\n        ]\n        assert result.outcome.stop_reason is StopReason.COMPLETE\n        assert EventType.AGENT_START in reached_second\n        assert len(failures) == 1\n        assert failures[0].extension_name == "broken-observer"\n        assert failures[0].diagnostic == "RuntimeError"\n\n    asyncio.run(scenario())\n\n\ndef test_extension_registers_a_chat_command_without_starting_a_run() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def greeting(arguments: str) -> str:\n                return f"hello {arguments}"\n\n            api.register_command("greet", greeting)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("must not run"), ModelEnd(StopReason.COMPLETE)]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/chat-command")),\n            extensions=[Extension("greeter", "1", configure)],\n        )\n\n        result = await session.execute_command("greet", "Omega")\n\n        assert result == "hello Omega"\n        assert adapter.received_requests == ()\n        assert session.busy is False\n\n    asyncio.run(scenario())\n\n\ndef test_extension_appends_a_versioned_namespaced_custom_entry() -> None:\n    store = MemorySessionStore(\n        id_factory=iter(("session-custom", "entry-custom")).__next__\n    )\n    state = store.create()\n\n    def configure(api) -> None:\n        api.append_custom_entry(\n            "stateful.cursor",\n            1,\n            {"position": 3},\n        )\n\n    AgentSession(\n        AgentRuntime(\n            ScriptedModelAdapter(\n                [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n            ),\n            ModelSpec("scripted/custom-entry"),\n        ),\n        store=store,\n        session_id=state.session_id,\n        extensions=[Extension("stateful", "1", configure)],\n    )\n\n    restored = store.read(state.session_id)\n    assert restored.history() == ()\n    assert len(restored.custom_entries()) == 1\n    entry = restored.custom_entries()[0]\n    assert entry.namespace == "stateful.cursor"\n    assert entry.version == 1\n    assert dict(entry.payload) == {"position": 3}\n\n\ndef test_run_snapshot_freezes_tools_until_the_next_run() -> None:\n    async def scenario() -> None:\n        old_calls = 0\n        new_calls = 0\n\n        async def old_handler(arguments: dict[str, object]) -> ToolResult:\n            nonlocal old_calls\n            old_calls += 1\n            return ToolResult("old")\n\n        async def new_handler(arguments: dict[str, object]) -> ToolResult:\n            nonlocal new_calls\n            new_calls += 1\n            return ToolResult("new")\n\n        class PausingAdapter:\n            def __init__(self) -> None:\n                self.started = asyncio.Event()\n                self.release = asyncio.Event()\n                self.turn = 0\n\n            async def stream(self, request):\n                self.turn += 1\n                if self.turn == 1:\n                    self.started.set()\n                    await self.release.wait()\n                if self.turn in {1, 3}:\n                    yield ToolCallDelta(0, f"call-{self.turn}", "versioned", "{}")\n                    yield ModelEnd(StopReason.TOOL_USE)\n                    return\n                yield TextDelta(f"complete-{self.turn}")\n                yield ModelEnd(StopReason.COMPLETE)\n\n        schema = {"type": "object"}\n        adapter = PausingAdapter()\n        runtime = AgentRuntime(\n            adapter,\n            ModelSpec("scripted/snapshot"),\n            tools=[Tool("versioned", "old behavior", schema, old_handler)],\n        )\n        session = AgentSession(runtime)\n\n        first = session.start("first")\n        await adapter.started.wait()\n        runtime.configure_tools(\n            [Tool("versioned", "new behavior", schema, new_handler)]\n        )\n        adapter.release.set()\n        first_result = await first.result()\n\n        second_result = await session.run("second")\n\n        assert old_calls == 1\n        assert new_calls == 1\n        assert first.snapshot.fingerprint == first_result.outcome.snapshot_fingerprint\n        assert first.snapshot.fingerprint != second_result.outcome.snapshot_fingerprint\n\n    asyncio.run(scenario())\n\n\ndef test_extension_adds_a_versioned_run_annotation() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def annotate(context):\n                api.add_run_annotation(\n                    context.snapshot.run_id,\n                    "reviewer.outcome",\n                    1,\n                    {"label": "accepted"},\n                )\n\n            api.register_hook(HookPoint.BEFORE_RUN, annotate)\n\n        session = AgentSession(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("annotated"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/annotation"),\n            ),\n            extensions=[Extension("reviewer", "1", configure)],\n        )\n\n        handle = session.start("review")\n        await handle.result()\n\n        annotations = session.run_annotations(handle.run_id)\n        assert len(annotations) == 1\n        assert annotations[0].namespace == "reviewer.outcome"\n        assert annotations[0].version == 1\n        assert dict(annotations[0].payload) == {"label": "accepted"}\n\n    asyncio.run(scenario())\n\n\ndef test_idle_reload_tears_down_before_constructing_named_replacement() -> None:\n    async def scenario() -> None:\n        torn_down = False\n        new_calls = 0\n\n        async def teardown_old() -> None:\n            nonlocal torn_down\n            torn_down = True\n\n        def configure_old(api) -> None:\n            return None\n\n        async def new_tool(arguments: dict[str, object]) -> ToolResult:\n            nonlocal new_calls\n            new_calls += 1\n            return ToolResult("new")\n\n        def configure_new(api) -> None:\n            assert torn_down is True\n            api.register_tool(\n                Tool("reloaded", "Reloaded Tool", {"type": "object"}, new_tool)\n            )\n\n        adapter = ScriptedModelAdapter(\n            [\n                [\n                    ToolCallDelta(0, "call-reload", "reloaded", "{}"),\n                    ModelEnd(StopReason.TOOL_USE),\n                ],\n                [TextDelta("reloaded"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/reload")),\n            extensions=[\n                Extension(\n                    "switcher",\n                    "1",\n                    configure_old,\n                    teardown=teardown_old,\n                )\n            ],\n        )\n\n        reload_result = await session.reload_extensions(\n            [Extension("switcher", "2", configure_new)]\n        )\n        run_result = await session.run("after reload")\n\n        assert reload_result.replaced_extensions == ("switcher",)\n        assert new_calls == 1\n        assert run_result.outcome.message.content[0].text == "reloaded"\n\n    asyncio.run(scenario())\n\n\ndef test_teardown_failure_is_a_warning_and_never_leaves_session_busy() -> None:\n    async def scenario() -> None:\n        async def broken_teardown() -> None:\n            raise RuntimeError("cleanup failed")\n\n        session = AgentSession(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/teardown-warning"),\n            ),\n            extensions=[\n                Extension(\n                    "fragile",\n                    "1",\n                    lambda api: None,\n                    source="explicit:fragile",\n                    teardown=broken_teardown,\n                )\n            ],\n        )\n\n        result = await session.reload_extensions([])\n\n        assert session.busy is False\n        assert len(result.warnings) == 1\n        assert result.warnings[0].source == "explicit:fragile"\n        assert result.warnings[0].phase == "teardown"\n        assert result.warnings[0].diagnostic == "RuntimeError"\n\n    asyncio.run(scenario())\n\n\ndef test_reload_is_rejected_while_a_run_is_active() -> None:\n    async def scenario() -> None:\n        class BlockingAdapter:\n            def __init__(self) -> None:\n                self.started = asyncio.Event()\n                self.release = asyncio.Event()\n\n            async def stream(self, request):\n                self.started.set()\n                await self.release.wait()\n                yield TextDelta("done")\n                yield ModelEnd(StopReason.COMPLETE)\n\n        adapter = BlockingAdapter()\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/busy-reload"))\n        )\n        handle = session.start("active")\n        await adapter.started.wait()\n\n        with pytest.raises(SessionBusyError, match="idle Session"):\n            await session.reload_extensions([])\n\n        adapter.release.set()\n        await handle.result()\n\n    asyncio.run(scenario())\n\n\ndef test_extension_startup_is_awaited_once_before_run_hooks() -> None:\n    async def scenario() -> None:\n        order: list[str] = []\n\n        async def startup() -> None:\n            await asyncio.sleep(0)\n            order.append("startup")\n\n        def configure(api) -> None:\n            async def before_run(context):\n                order.append("before_run")\n\n            api.register_hook(HookPoint.BEFORE_RUN, before_run)\n\n        session = AgentSession(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [\n                        [TextDelta("first"), ModelEnd(StopReason.COMPLETE)],\n                        [TextDelta("second"), ModelEnd(StopReason.COMPLETE)],\n                    ]\n                ),\n                ModelSpec("scripted/startup"),\n            ),\n            extensions=[\n                Extension("lifecycle", "1", configure, startup=startup)\n            ],\n        )\n\n        await session.run("one")\n        await session.run("two")\n\n        assert order == ["startup", "before_run", "before_run"]\n\n    asyncio.run(scenario())\n\n\ndef test_tool_and_command_name_replacement_must_be_explicit_and_is_recorded() -> None:\n    async def base(arguments: dict[str, object]) -> ToolResult:\n        return ToolResult("base")\n\n    async def replacement(arguments: dict[str, object]) -> ToolResult:\n        return ToolResult("replacement")\n\n    async def first_command(arguments: str) -> str:\n        return "first"\n\n    async def second_command(arguments: str) -> str:\n        return "second"\n\n    def configure_first(api) -> None:\n        api.register_command("status", first_command)\n\n    def configure_second(api) -> None:\n        api.register_tool(\n            Tool("shared", "Replacement", {"type": "object"}, replacement),\n            replace=True,\n        )\n        api.register_command("status", second_command, replace=True)\n\n    session = AgentSession(\n        AgentRuntime(\n            ScriptedModelAdapter(\n                [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n            ),\n            ModelSpec("scripted/replacements"),\n            tools=[Tool("shared", "Base", {"type": "object"}, base)],\n        ),\n        extensions=[\n            Extension("first", "1", configure_first),\n            Extension("second", "1", configure_second),\n        ],\n    )\n\n    assert [\n        (record.kind, record.name, record.previous_owner, record.replacement_owner)\n        for record in session.extension_replacements\n    ] == [\n        ("tool", "shared", "runtime", "second"),\n        ("command", "status", "first", "second"),\n    ]\n\n\ndef test_extension_initialization_failure_prevents_session_construction() -> None:\n    def broken(api) -> None:\n        raise RuntimeError("bad configuration")\n\n    with pytest.raises(ExtensionInitializationError) as captured:\n        AgentSession(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec("scripted/init-failure"),\n            ),\n            extensions=[\n                Extension(\n                    "broken",\n                    "1",\n                    broken,\n                    source="explicit:/trusted/broken.py",\n                )\n            ],\n        )\n\n    assert captured.value.source == "explicit:/trusted/broken.py"\n    assert "RuntimeError" in str(captured.value)\n\n\ndef test_custom_entries_round_trip_through_jsonl_without_entering_model_history(\n    tmp_path,\n) -> None:\n    store = JSONLSessionStore(\n        tmp_path,\n        id_factory=iter(("session-jsonl-custom", "entry-jsonl-custom")).__next__,\n    )\n    state = store.create()\n\n    def configure(api) -> None:\n        api.append_custom_entry("durable.state", 2, {"enabled": True})\n\n    AgentSession(\n        AgentRuntime(\n            ScriptedModelAdapter(\n                [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n            ),\n            ModelSpec("scripted/jsonl-custom"),\n        ),\n        store=store,\n        session_id=state.session_id,\n        extensions=[Extension("durable", "1", configure)],\n    )\n\n    restored = JSONLSessionStore(tmp_path).read(state.session_id)\n    assert restored.history() == ()\n    assert restored.effective_history() == ()\n    assert restored.custom_entries()[0].version == 2\n    assert dict(restored.custom_entries()[0].payload) == {"enabled": True}\n\n\ndef test_run_snapshot_and_events_retain_effective_configuration_evidence() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def inspect(context):\n                return None\n\n            api.register_hook(HookPoint.BEFORE_RUN, inspect)\n\n        retry = RetryPolicy(delays=(0.25,))\n        guard = RunGuard(max_turns=3)\n        compaction = CompactionPolicy(\n            reserve_tokens=32,\n            keep_recent_tokens=16,\n        )\n        session = AgentSession(\n            AgentRuntime(\n                ScriptedModelAdapter(\n                    [TextDelta("evidence"), ModelEnd(StopReason.COMPLETE)]\n                ),\n                ModelSpec(\n                    "scripted/evidence",\n                    context_window=1024,\n                    max_output_tokens=128,\n                ),\n                retry_policy=retry,\n                run_guard=guard,\n                generation_settings={"temperature": 0},\n            ),\n            compaction_policy=compaction,\n            extensions=[Extension("evidence", "2", configure)],\n            prompt_hashes={"system": "sha256:prompt"},\n            resource_hashes={"skill:review": "sha256:skill"},\n        )\n\n        handle = session.start("capture")\n        await handle.result()\n        events = [event async for event in handle.events()]\n        snapshot = handle.snapshot\n\n        assert snapshot.model.model_id == "scripted/evidence"\n        assert snapshot.extension_identities == ("evidence@2",)\n        assert snapshot.retry_policy is retry\n        assert snapshot.run_guard is guard\n        assert snapshot.compaction_policy is compaction\n        assert dict(snapshot.generation_settings) == {"temperature": 0}\n        assert snapshot.prompt_hashes == (("system", "sha256:prompt"),)\n        assert snapshot.resource_hashes == (\n            ("skill:review", "sha256:skill"),\n        )\n        assert all(event.run_id == handle.run_id for event in events)\n        assert all(\n            event.snapshot_fingerprint == snapshot.fingerprint for event in events\n        )\n\n    asyncio.run(scenario())\n\n\ndef test_before_run_context_replacement_is_local_to_each_run() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def add_transient_context(context):\n                return (\n                    *context.value,\n                    AgentMessage.text(Role.SYSTEM, "transient extension context"),\n                )\n\n            api.register_hook(HookPoint.BEFORE_RUN, add_transient_context)\n\n        adapter = ScriptedModelAdapter(\n            [\n                [TextDelta("first"), ModelEnd(StopReason.COMPLETE)],\n                [TextDelta("second"), ModelEnd(StopReason.COMPLETE)],\n            ]\n        )\n        runtime = AgentRuntime(adapter, ModelSpec("scripted/run-local-context"))\n        session = AgentSession(\n            runtime,\n            extensions=[Extension("transient", "1", configure)],\n        )\n\n        await session.run("one")\n        await session.run("two")\n\n        for request in adapter.received_requests:\n            transient = [\n                message\n                for message in request.messages\n                if message.role is Role.SYSTEM\n                and message.content[0].text == "transient extension context"\n            ]\n            assert len(transient) == 1\n        assert all(\n            message.role is not Role.SYSTEM\n            for message in runtime.history\n            if isinstance(message, AgentMessage)\n        )\n\n    asyncio.run(scenario())\n\n\ndef test_before_model_hook_failure_terminates_before_provider_io() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def fail(context):\n                raise RuntimeError("request review failed")\n\n            api.register_hook(HookPoint.BEFORE_MODEL_REQUEST, fail)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("must not run"), ModelEnd(StopReason.COMPLETE)]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/model-hook-failure")),\n            extensions=[Extension("request-review", "1", configure)],\n        )\n\n        result = await session.run("review")\n\n        assert result.outcome.stop_reason is StopReason.ERROR\n        assert result.outcome.error is not None\n        assert result.outcome.error.code.value == "hook_failed"\n        assert adapter.received_requests == ()\n\n    asyncio.run(scenario())\n\n\ndef test_chat_command_failure_is_isolated_without_starting_a_run() -> None:\n    async def scenario() -> None:\n        def configure(api) -> None:\n            async def broken(arguments: str) -> object:\n                raise RuntimeError("command failed")\n\n            api.register_command("broken", broken)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("must not run"), ModelEnd(StopReason.COMPLETE)]\n        )\n        session = AgentSession(\n            AgentRuntime(adapter, ModelSpec("scripted/command-failure")),\n            extensions=[Extension("commands", "1", configure)],\n        )\n\n        with pytest.raises(CommandExecutionError, match="/broken"):\n            await session.execute_command("broken")\n\n        assert adapter.received_requests == ()\n        assert session.busy is False\n\n    asyncio.run(scenario())\n\n\ndef test_compaction_hook_failure_warns_at_threshold_but_is_terminal_for_overflow() -> None:\n    async def threshold_scenario() -> None:\n        def configure(api) -> None:\n            async def reject(context):\n                raise RuntimeError("compaction review failed")\n\n            api.register_hook(HookPoint.BEFORE_COMPACTION, reject)\n\n        adapter = ScriptedModelAdapter(\n            [TextDelta("settled"), ModelEnd(StopReason.COMPLETE)]\n        )\n        session = AgentSession(\n            AgentRuntime(\n                adapter,\n                ModelSpec(\n                    "scripted/threshold-hook-failure",\n                    context_window=8,\n                    max_output_tokens=1,\n                ),\n            ),\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            extensions=[Extension("compaction-review", "1", configure)],\n        )\n\n        result = await session.run("a prompt long enough for threshold compaction")\n\n        assert result.outcome.stop_reason is StopReason.COMPLETE\n        assert session.warnings[-1].code is CompactionWarningCode.THRESHOLD_FAILED\n        assert len(adapter.received_requests) == 1\n\n    async def overflow_scenario() -> None:\n        class OverflowAdapter:\n            async def stream(self, request):\n                raise ModelAdapterError(\n                    ModelError(\n                        ModelErrorCode.CONTEXT_OVERFLOW,\n                        "too large",\n                        False,\n                    )\n                )\n                yield  # pragma: no cover\n\n        def configure(api) -> None:\n            async def reject(context):\n                raise RuntimeError("compaction review failed")\n\n            api.register_hook(HookPoint.BEFORE_COMPACTION, reject)\n\n        runtime = AgentRuntime(\n            OverflowAdapter(),\n            ModelSpec("scripted/overflow-hook-failure"),\n            history=(\n                AgentMessage.text(Role.USER, "old question"),\n                AgentMessage.text(Role.ASSISTANT, "old answer"),\n            ),\n        )\n        session = AgentSession(\n            runtime,\n            compaction_policy=CompactionPolicy(keep_recent_tokens=1),\n            extensions=[Extension("compaction-review", "1", configure)],\n        )\n\n        result = await session.run("continue")\n\n        assert result.outcome.stop_reason is StopReason.ERROR\n        assert result.outcome.error is not None\n        assert result.outcome.error.code is ModelErrorCode.COMPACTION_FAILED\n\n    asyncio.run(threshold_scenario())\n    asyncio.run(overflow_scenario())\n'


In [ ]:
CHAPTER_CONTRACT_TEST_SOURCE = 'from __future__ import annotations\n\nimport pytest\n\nfrom agent_harness import (\n    AgentRuntime,\n    AgentSession,\n    ExtensionAPI,\n    HookPoint,\n    ModelEnd,\n    ModelSpec,\n    ScriptedModelAdapter,\n    StopReason,\n    TextDelta,\n)\n\n\ndef test_version_one_exposes_only_the_bounded_hook_set() -> None:\n    assert tuple(point.value for point in HookPoint) == (\n        "before_run",\n        "before_model_request",\n        "before_tool_call",\n        "after_tool_call",\n        "before_compaction",\n    )\n    for excluded_surface in (\n        "discover",\n        "install",\n        "register_provider",\n        "register_cli_argument",\n        "register_tui_component",\n        "start_background_service",\n    ):\n        assert not hasattr(ExtensionAPI, excluded_surface)\n\n\ndef test_core_accepts_explicit_extension_values_not_paths_or_packages() -> None:\n    runtime = AgentRuntime(\n        ScriptedModelAdapter(\n            [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n        ),\n        ModelSpec("scripted/explicit-only"),\n    )\n\n    with pytest.raises(TypeError, match="Extension values"):\n        AgentSession(runtime, extensions=["project_extension.py"])  # type: ignore[list-item]\n'


In [ ]:
PYPROJECT_SOURCE = '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "agent-harness"\nversion = "0.7.0"\ndescription = "Chapter 7 bounded Events, Hooks, Extensions, and Run Snapshots"\nreadme = "README.md"\nrequires-python = ">=3.11"\ndependencies = ["jsonschema>=4.23,<5", "openai>=1.40,<3"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.setuptools.package-data]\nagent_harness = ["py.typed"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n'


In [ ]:
README_SOURCE = '# Agent Harness — Chapter 7 Checkpoint\n\nThis cumulative Checkpoint adds a bounded, explicitly supplied Extension surface. An `Extension` configures only its public `ExtensionAPI`: Tools with explicit replacement, ordered passive Event subscribers, the five declared Hooks, chat commands, versioned namespaced Custom Entries, Run Annotations, and awaited lifecycle handlers. Core code performs no discovery, installation, or implicit project loading.\n\nHigh-level subscribers are awaited in registration order. `message_end` remains a persistence barrier before dependent work, while `agent_end` settles before `AgentSession.busy` becomes false. Subscriber failures produce correlated `subscriber_failed` observations and do not change semantic results. `AgentRunHandle.events()` and `SessionRunHandle.events()` remain ordered observational streams whose consumers do not delay Runtime progress.\n\nHooks fail safe at each intervention seam. Run and model-request failures terminate without uncertain model work, Tool preflight failure blocks execution, Tool-result failure substitutes a safe error, and Compaction failure cancels the operation unless normalized overflow makes recovery terminal. Hook order and effective registrations are frozen in the accepted Run Snapshot.\n\n`RunSnapshot` captures the effective ModelSpec, generation settings, Tools, Extension identities, Hook order, retry, Compaction and guard configuration, prompt and resource hashes, plus a stable fingerprint. Events and outcomes retain Run and snapshot correlation. Runtime reconfiguration can therefore affect only later accepted Runs.\n\nIdle Extension reload awaits teardown before constructing replacements. Initialization identifies its explicit source; teardown failure becomes a warning and cannot leave the Session busy. Custom Entries persist in the Session tree without entering model history, while Run Annotations remain a separate in-memory seam for the later observability layer.\n'


In [ ]:
sys.path.remove(temporary_package.name)
temporary_package.cleanup()
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]


In [ ]:
from course.tools.checkpoint import checkpoint_drift, export_checkpoint

checkpoint_result = export_checkpoint(
    ROOT / 'course' / 'notebooks' / '07_extensions.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch07',
)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
assert checkpoint_drift(
    ROOT / 'course' / 'notebooks' / '07_extensions.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch07',
) == ()
checkpoint_result


## Public API Summary

Supply trusted `Extension` values to `AgentSession` or `create_agent_session`. Configure them through `ExtensionAPI`; inspect `SessionRunHandle.snapshot`, correlated Events, Custom Entries, Run Annotations, explicit replacement records, and reload warnings. Use `await session.reload_extensions(...)` only while idle and `await session.execute_command(...)` for registered chat commands.